# Поиск объявлений услуг по пользовательскому запросу

Задача - выбрать до 50 объявлений из корпуса для каждого запроса. Качество оценивается по **Recall@50**: доле известных релевантных объявлений, попавших в ответ, с усреднением по запросам. Порядок этих 50 объявлений на итоговую метрику не влияет.

Решение объединяет символьный и словесный TF-IDF, географию, эмбеддинги E5 и две модели CatBoost. Ниже последовательно показаны эксперименты и пять отправок. Лучший результат на Stepik - **0.773831**.

### Запуск

Ячейки выполняются сверху вниз. Пути к трем Parquet-файлам задаются в первой кодовой ячейке, `OUTPUT_DIR` - при создании первого ответа. Для запуска вне Kaggle эти пути нужно заменить. Промежуточные отправки перезаписывают answer.csv; последняя кодовая ячейка создает итоговый вариант.

Используются Python, pandas, NumPy, scikit-learn, CatBoost, PyTorch, Transformers, tqdm, IPython и движок чтения Parquet, например pyarrow. Расчеты выполнены в Kaggle с двумя NVIDIA T4; текущий код требует CUDA. E5 загружается из Hugging Face при отсутствии локального кэша, после чего вычисления выполняются локально. Для запуска без сети модель и зависимости нужно подготовить заранее. Использованные методы: TF-IDF, объединение по обратным рангам (RRF), CatBoostRanker с YetiRank и открытая модель [intfloat/multilingual-e5-small](https://huggingface.co/intfloat/multilingual-e5-small). E5 применяется без дообучения.


## 1. Исследование данных

### 1.1. Загрузка и структура таблиц

Загружаем обучающие взаимодействия, запросы бенчмарка и корпус объявлений. Смотрим размеры, типы колонок, пропуски и примеры строк. Идентификаторы сохраняем в исходном виде.


In [1]:
import pandas as pd
from IPython.display import display

TRAIN_PATH = "/kaggle/input/datasets/nikitanazarov1504/nlp-llm-avito/dataset/train.parquet"
QUERIES_PATH = "/kaggle/input/datasets/nikitanazarov1504/nlp-llm-avito/dataset/benchmark_queries.parquet"
ITEMS_PATH = "/kaggle/input/datasets/nikitanazarov1504/nlp-llm-avito/dataset/benchmark_items.parquet"

train = pd.read_parquet(TRAIN_PATH)
queries = pd.read_parquet(QUERIES_PATH)
items = pd.read_parquet(ITEMS_PATH)

tables = {
    "train": train,
    "benchmark_queries": queries,
    "benchmark_items": items,
}

for name, df in tables.items():
    print(f"\n{name}: {df.shape[0]:,} строк, {df.shape[1]} колонок")
    print(f"Память: {round(df.memory_usage(deep=True).sum() / 1024**2, 1)} МБ")

    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_pct": (100 * df.isna().mean()).round(2),
    }))

    display(df.head(3))


train: 497,673 строк, 19 колонок
Память: 3647.2 МБ


,dtype,missing_count,missing_pct
search_query,object,0,0.00
search_location_id,int64,0,0.00
search_is_delivery_search,int32,0,0.00
search_infm_params_text,object,0,0.00
search_category,int64,0,0.00
item_title_raw,object,0,0.00
item_rating_reviews_count,float64,19211,3.86
item_rating,float64,28232,5.67
item_price,object,0,0.00
item_microcat_id,int64,0,0.00


,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,скупка телевизоров,652430,0,,114,Скупка б/у техники,91.0,4.989011,1.000000000000000,2303374,39.712818145751953,652430,54.629230499267578,True,False,Вид услуги Место оказания услуг Первомайский п...,e8b685dffe1a408e,Скупаю практически любую современную новую и б...,114
1,автоподбор,640860,0,Рейтинг пользователя 4 звезды и выше,114,Автоподбор Разовый осмотр автомобиля,462.0,4.982684,3500.000000000000000,2303374,44.051009999999998,640860,56.273389999999999,False,False,Вид услуги Место оказания услуг Нижний Новгоро...,92e1b0446f827b59,🚗 Автоподбор и выездная диагностика автомобиля...,114
2,баня на дровах,653240,0,"Онлайн-запись Тип услуги СПА-услуги, массаж Ви...",114,"Баня на дровах ""Прованс"" на Цветочной",NaN,NaN,1600.000000000000000,86469,30.128784180000000,653240,59.783687590000000,False,False,"Вид услуги Красота, здоровье Место оказания ус...",624846856ce81d69,В ритме современной жизни так сложно найти мо...,114



benchmark_queries: 2,452 строк, 6 колонок
Память: 0.7 МБ


,dtype,missing_count,missing_pct
query_id,object,0,0.0
search_query,object,0,0.0
search_location_id,int64,0,0.0
search_is_delivery_search,int32,0,0.0
search_infm_params_text,object,0,0.0
search_category,int64,0,0.0


,query_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category
0,70DfDUpwjxB4lzFd,перевозки владикавказ тбилиси,649820,0,,114
1,JTrdTaZJvSiLPkXj,обзвон по базе,107620,0,Вид услуги Деловые услуги,114
2,LZCZNoVG4AFUkVRJ,липоредукция подбородка,637640,0,"Вид услуги Красота, здоровье",114



benchmark_items: 189,212 строк, 14 колонок
Память: 1362.3 МБ


,dtype,missing_count,missing_pct
item_title_raw,object,0,0.00
item_rating_reviews_count,float64,11615,6.14
item_rating,float64,17631,9.32
item_price,object,0,0.00
item_microcat_id,int64,0,0.00
item_longitude,object,1,0.00
item_location_id,int64,0,0.00
item_latitude,object,1,0.00
item_is_phone_hidden,bool,0,0.00
item_is_message_forbidden,bool,0,0.00


,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,Ремонт/выкуп компьют. и ноутбуков с выездом на...,57.0,5.0,500.000000000000000,2097573,42.047193961004901,631060,44.228180450924000,False,False,Вид услуги Компьютерная помощь Место оказания ...,111eb8b979577d79,"Выезд на дом в любое время, вплоть до 23:00\n\...",114
1,Обучение ребенка чтению,7.0,5.0,900.000000000000000,86453,49.627391820000000,631870,58.627609249999999,False,False,"Вид услуги Обучение, курсы Место оказания услу...",75fc8e10f5a66fc4,"Дорогие родители и маленькие книголюбы! \n\nЯ,...",114
2,Афрокудри 5+,14.0,5.0,1000.000000000000000,86467,40.936892000000000,628500,56.990307000000001,True,False,"Вид услуги Красота, здоровье Место оказания ус...",3f6ae81704565b9c,ВНИМАНИЕ Укладка !!!! 🥳\n\nАФРОкудри отличный ...,114


### 1.2. Повторы и пересечения

Проверяем идентификаторы и считаем уникальные тексты запросов, сочетания их признаков и выбранные объявления. Это помогает отличить повтор одного взаимодействия от разных запросов с одинаковым текстом.


In [2]:
for table_name, df, id_col in (("benchmark_queries", queries, "query_id"), ("benchmark_items", items, "item_id")):
    print(f"\n{table_name} / {id_col}")
    print("Пропусков:", df[id_col].isna().sum())
    print("Повторов:", df[id_col].duplicated().sum())
    print("Типы значений:")
    print(df[id_col].map(lambda x: type(x).__name__).value_counts())
    print("Длины идентификаторов:")
    print(df[id_col].dropna().astype(str).str.len().value_counts())

query_features = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]

for name, df in (("train", train), ("benchmark_queries", queries)):
    missing = set(query_features) - set(df.columns)
    if missing:
        raise ValueError(f"В {name} отсутствуют колонки: {sorted(missing)}")

    print(f"\n{name}")
    print("Уникальных текстов:", df["search_query"].nunique(dropna=False))
    print(
        "Уникальных сочетаний признаков запроса:",
        len(df[query_features].drop_duplicates()),
    )

if "item_id" in train.columns:
    positive_counts = (
        train.groupby(query_features, dropna=False)["item_id"].nunique()
    )

    print("\nКоличество уникальных выбранных объявлений на сочетание признаков:")
    display(positive_counts.describe(percentiles=[0.5, 0.9, 0.99]))

    print(
        "Повторов пары «признаки запроса — item_id»:",
        train.duplicated(query_features + ["item_id"]).sum(),
    )

    train_item_ids = set(train["item_id"].dropna())
    benchmark_item_ids = set(items["item_id"].dropna())

    print("Уникальных item_id в train:", len(train_item_ids))
    print(
        "Общих item_id у train и корпуса бенчмарка:",
        len(train_item_ids & benchmark_item_ids),
    )

known_query_texts = set(train["search_query"].dropna())
print(
    "\nДоля запросов бенчмарка с текстом, встречавшимся в train:",
    f"{queries['search_query'].isin(known_query_texts).mean():.2%}",
)


benchmark_queries / query_id
Пропусков: 0
Повторов: 0
Типы значений:
query_id
str    2452
Name: count, dtype: int64
Длины идентификаторов:
query_id
16    2452
Name: count, dtype: int64

benchmark_items / item_id
Пропусков: 0
Повторов: 0
Типы значений:
item_id
str    189212
Name: count, dtype: int64
Длины идентификаторов:
item_id
16    189212
Name: count, dtype: int64

train
Уникальных текстов: 74529
Уникальных сочетаний признаков запроса: 354463

benchmark_queries
Уникальных текстов: 2452
Уникальных сочетаний признаков запроса: 2452

Количество уникальных выбранных объявлений на сочетание признаков:


count    354463.000000
mean          1.317638
std           1.611180
min           1.000000
50%           1.000000
90%           2.000000
99%           6.000000
max         278.000000
Name: item_id, dtype: float64

Повторов пары «признаки запроса — item_id»: 30619
Уникальных item_id в train: 344825
Общих item_id у train и корпуса бенчмарка: 18142

Доля запросов бенчмарка с текстом, встречавшимся в train: 36.99%


### 1.3. Нормализация и покрытие корпусом

Приводим текст к нижнему регистру, заменяем `е` на `е` и нормализуем пробелы. Удаляем повторы положительных пар и проверяем, сколько выбранных объявлений есть в корпусе бенчмарка. После нормализации в train встречается **37.36%** текстов бенчмарка; объявления из его корпуса покрывают **6.33%** уникальных положительных пар train.


In [3]:
import re
import unicodedata
import numpy as np

def normalize_query(value):
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = text.lower().replace("ё", "е")
    return re.sub(r"\s+", " ", text).strip()

query_features = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]

pairs = (
    train[query_features + ["item_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

pairs["query_norm"] = pairs["search_query"].map(normalize_query)
benchmark_query_norm = queries["search_query"].map(normalize_query)

known_texts = set(pairs["query_norm"])
known_items = set(items["item_id"])

pairs["in_benchmark_corpus"] = pairs["item_id"].isin(known_items)

print("Пересечение текстов после нормализации:")
print(f"{benchmark_query_norm.isin(known_texts).mean():.2%}")

print("\nПоложительных пар после удаления повторов:", len(pairs))
print(
    "Из них с объявлением в корпусе бенчмарка:",
    int(pairs["in_benchmark_corpus"].sum()),
)
print(
    "Доля таких пар:",
    f"{pairs['in_benchmark_corpus'].mean():.2%}",
)

coverage = (
    pairs.groupby(query_features, dropna=False)
    .agg(
        n_positive=("item_id", "size"),
        n_in_corpus=("in_benchmark_corpus", "sum"),
    )
    .reset_index()
)

print("\nСочетаний признаков запроса:")
print("Всего:", len(coverage))
print("Есть хотя бы один позитив в корпусе:",
      int((coverage["n_in_corpus"] > 0).sum()))
print("Все позитивы есть в корпусе:",
      int((coverage["n_in_corpus"] == coverage["n_positive"]).sum()))

print(
    "\nУникальных нормализованных текстов с позитивами в корпусе:",
    pairs.loc[pairs["in_benchmark_corpus"], "query_norm"].nunique(),
)

print("\nСамые частые тексты по числу уникальных положительных пар:")
display(pairs["query_norm"].value_counts().head(15))

Пересечение текстов после нормализации:
37.36%

Положительных пар после удаления повторов: 467054
Из них с объявлением в корпусе бенчмарка: 29572
Доля таких пар: 6.33%

Сочетаний признаков запроса:
Всего: 354463
Есть хотя бы один позитив в корпусе: 26556
Все позитивы есть в корпусе: 18436

Уникальных нормализованных текстов с позитивами в корпусе: 12174

Самые частые тексты по числу уникальных положительных пар:


query_norm
маникюр                    6015
массаж                     4969
наращивание ресниц         4959
установка кондиционеров    3518
педикюр                    3128
сантехник                  2975
электрик                   2816
вывоз мусора               2687
эвакуатор                  2547
покос травы                2265
автоэлектрик               2168
манипулятор                1934
вспашка земли              1878
наращивание ногтей         1849
грузоперевозки газель      1697
Name: count, dtype: int64

### 1.4. География и заполненность признаков

Изучаем совпадение локаций в положительных парах, категории и длины текстовых полей. Пустые строки учитываем отдельно от `NaN`: отсутствие пропусков в таблице еще не означает, что текст заполнен.


In [4]:
unique_pair_mask = ~train.duplicated(query_features + ["item_id"])

meta_columns = [
    "search_location_id",
    "item_location_id",
    "search_category",
    "item_category_id",
    "search_is_delivery_search",
]

meta = train.loc[unique_pair_mask, meta_columns].copy()
meta["same_location"] = (meta["search_location_id"] == meta["item_location_id"])
meta["same_category"] = (meta["search_category"] == meta["item_category_id"])

print("Доля положительных пар с совпадающей локацией:")
print(f"{meta['same_location'].mean():.2%}")

print("\nСовпадение локации в зависимости от флага доставки:")
display(
    meta.groupby("search_is_delivery_search")
    .agg(
        n_pairs=("same_location", "size"),
        same_location_share=("same_location", "mean"),
    )
)

print("\nСамые частые пары категорий:")
display(
    meta.groupby(["search_category", "item_category_id"])
    .size()
    .sort_values(ascending=False)
    .head(15)
    .rename("n_pairs")
    .reset_index()
)

print("\nКоличество уникальных локаций:")
print("В запросах train:", train["search_location_id"].nunique())
print("В запросах бенчмарка:", queries["search_location_id"].nunique())
print("В корпусе объявлений:", items["item_location_id"].nunique())

print("\nДоля запросов бенчмарка с локацией, отсутствующей в train:")
print(f"{(~queries['search_location_id'].isin(train['search_location_id'])).mean():.2%}")

text_stats = []

for table_name, df, columns in (
    ("train", train, ["search_query", "search_infm_params_text"]),
    ("queries", queries, ["search_query", "search_infm_params_text"]),
    ("items", items, [
        "item_title_raw",
        "item_description_raw",
        "item_infm_params_text",
    ]),
):
    for column in columns:
        lengths = df[column].fillna("").astype(str).str.strip().str.len()
        text_stats.append({
            "table": table_name,
            "column": column,
            "empty_pct": round(100 * lengths.eq(0).mean(), 2),
            "median_chars": lengths.median(),
            "p95_chars": lengths.quantile(0.95),
            "max_chars": lengths.max(),
        })

print("\nЗаполненность и длины текстовых полей:")
display(pd.DataFrame(text_stats))

Доля положительных пар с совпадающей локацией:
82.20%

Совпадение локации в зависимости от флага доставки:


,n_pairs,same_location_share
search_is_delivery_search,,
0,467042,0.821977
1,12,0.500000



Самые частые пары категорий:


,search_category,item_category_id,n_pairs
0,114,114,467002
1,0,114,34
2,114,27,3
3,81,114,2
4,10,114,2
5,114,28,2
6,114,19,2
7,114,81,2
8,114,40,2
9,114,112,2



Количество уникальных локаций:
В запросах train: 2782
В запросах бенчмарка: 273
В корпусе объявлений: 2877

Доля запросов бенчмарка с локацией, отсутствующей в train:
0.00%

Заполненность и длины текстовых полей:


,table,column,empty_pct,median_chars,p95_chars,max_chars
0,train,search_query,0.00,17.0,31.0,113
1,train,search_infm_params_text,32.95,28.0,97.0,1079
2,queries,search_query,0.00,22.0,39.0,70
3,queries,search_infm_params_text,63.05,0.0,64.0,217
4,items,item_title_raw,0.00,35.0,50.0,100
5,items,item_description_raw,0.02,1054.0,4205.0,8494
6,items,item_infm_params_text,0.00,738.0,2475.0,19068


### Выводы по данным

- Локации совпадают в **82.20%** положительных пар. География полезна, но жесткое ограничение той же локацией исключило бы оставшиеся 17.80% пар.
- Почти все взаимодействия относятся к категории 114, поэтому категория сама по себе слабо различает объявления.
- Пересечение train с корпусом бенчмарка небольшое. Для локальной проверки нужен корпус, содержащий все отложенные позитивы.
- Большинство текстов бенчмарка не встречается в train. Основную валидацию строим на отложенных текстах.


## 2. Локальная валидация

### 2.1. Разделение по текстам запросов

С фиксированным seed выбираем **2 000 нормализованных текстов** и по одному случайному контексту для каждого: формулировку, локацию, фильтры, категорию и доставку. Сохраняем все известные позитивы выбранного контекста. Все взаимодействия с отложенными текстами исключаем из обучения CatBoost.

Такая проверка измеряет качество на новых текстах. Она не полностью повторяет бенчмарк, где часть текстов знакома по train.


In [5]:
import numpy as np
import pandas as pd

SEED = 42
N_VALID = 2_000

query_features = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]

contexts = pairs[query_features + ["query_norm"]].drop_duplicates()

# выбираем тексты равновероятно, чтобы частые запросы не получали больший шанс попасть в проверочную выборку
rng = np.random.default_rng(SEED)
available_texts = np.array(sorted(contexts["query_norm"].unique()))

valid_texts = set(
    rng.choice(
        available_texts,
        size=min(N_VALID, len(available_texts)),
        replace=False,
    )
)

# для каждого выбранного текста случайно берём один контекст: исходную формулировку, локацию, фильтры, категорию и доставку
valid_queries = (
    contexts[contexts["query_norm"].isin(valid_texts)]
    .sample(frac=1, random_state=SEED)
    .drop_duplicates("query_norm")
    .reset_index(drop=True)
)

valid_queries.insert(
    0, "val_id", np.arange(len(valid_queries), dtype=np.int64)
)

# собираем все известные позитивы выбранного контекста
valid_positive_rows = valid_queries[
    ["val_id"] + query_features
].merge(
    pairs[query_features + ["item_id"]],
    on=query_features,
    how="inner",
    validate="one_to_many",
)

valid_targets = (
    valid_positive_rows.groupby("val_id")["item_id"]
    .agg(set)
    .to_dict()
)

# удаляем все контексты отложенных текстов, а не только выбранные
fit_pairs = pairs.loc[
    ~pairs["query_norm"].isin(valid_texts),
    query_features + ["query_norm", "item_id"],
].copy()

assert not set(fit_pairs["query_norm"]) & valid_texts
assert len(valid_targets) == len(valid_queries)
assert all(len(targets) > 0 for targets in valid_targets.values())

print("Проверочных запросов:", len(valid_queries))
print("Положительных пар для будущего обучения:", len(fit_pairs))
print("Количество позитивов на проверочный запрос:")
display(pd.Series([len(x) for x in valid_targets.values()]).describe())

Проверочных запросов: 2000
Положительных пар для будущего обучения: 449103
Количество позитивов на проверочный запрос:


count    2000.000000
mean        1.062000
std         0.334981
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         8.000000
dtype: float64

### 2.2. Поисковый корпус

Объединяем объявления бенчмарка и train: получается **515 895 уникальных объявлений**. При повторе `item_id` используем запись из бенчмарка. В этом корпусе доступны все проверочные позитивы и множество конкурирующих объявлений.

Тексты документов используются для построения индексов, но связи отложенных запросов с объявлениями не участвуют в обучении. При подготовке ответов для Stepik поиск ограничивается исходными **189 212 объявлениями бенчмарка**. Из-за различий корпусов локальный балл не равен ожидаемому баллу Stepik.


In [6]:
# пока нужны только заголовок, локация и идентификатор (тяжелые описания не копируем)
corpus_columns = [
    "item_id",
    "item_title_raw",
    "item_location_id",
]

corpus = (
    pd.concat(
        [
            items[corpus_columns],
            train[corpus_columns],
        ],
        ignore_index=True,
    )
    # при наличии объявления в обоих файлах используем его представление из benchmark_items
    .drop_duplicates("item_id", keep="first")
    .reset_index(drop=True)
)

corpus_ids = corpus["item_id"].to_numpy()
corpus_locations = corpus["item_location_id"].to_numpy()

all_valid_positive_ids = set().union(*valid_targets.values())
assert all_valid_positive_ids.issubset(set(corpus_ids))

print("Объявлений в проверочном корпусе:", len(corpus))
print("Все проверочные позитивы присутствуют в корпусе")

Объявлений в проверочном корпусе: 515895
Все проверочные позитивы присутствуют в корпусе


## 3. Эксперимент: символьный поиск по заголовкам

### 3.1. Индекс TF-IDF

Представляем заголовки символьными n-граммами длиной 3-5. Такой поиск учитывает частичные совпадения слов и похожие написания. Ограничиваем словарь 150 тысячами признаков и храним матрицу в разреженном формате.


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from time import perf_counter

started = perf_counter()

corpus_titles = corpus["item_title_raw"].map(normalize_query)

vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=150_000,
    sublinear_tf=True,
    norm="l2",
    dtype=np.float32,
)

# строим словарь по документам поискового корпуса
X_items = vectorizer.fit_transform(corpus_titles)
X_valid = vectorizer.transform(valid_queries["query_norm"])

X_items_T = X_items.T.tocsc()

matrix_mb = (
    X_items.data.nbytes
    + X_items.indices.nbytes
    + X_items.indptr.nbytes
) / 1024**2

print(f"Размер матрицы объявлений: {X_items.shape}")
print(f"Память основной разреженной матрицы: {matrix_mb:.1f} МБ")
print(f"Время построения: {perf_counter() - started:.1f} сек.")
print(f"Запросов без известных индексу сочетаний символов: {int((X_valid.getnnz(axis=1) == 0).sum())}")

Размер матрицы объявлений: (515895, 150000)
Память основной разреженной матрицы: 288.8 МБ
Время построения: 27.0 сек.
Запросов без известных индексу сочетаний символов: 0


### 3.2. Влияние локации

Сравниваем текстовое сходство с вариантом, где совпадающая локация добавляет к оценке 0.15. Бонус получают только объявления с ненулевым текстовым сходством. Recall считаем по каждому запросу, затем усредняем.

**Результат сохраненного запуска:** 0.195704 без географии и **0.325929** с ней. Географию включаем в следующие эксперименты.


In [8]:
from time import perf_counter
from tqdm.auto import tqdm
import numpy as np
import pandas as pd

started = perf_counter()
X_items_T = X_items.T.tocsr()
print(f"Подготовка матрицы: {perf_counter() - started:.1f} сек.")

geo_bonuses = {
    "title_char_tfidf": 0.0,
    "title_char_tfidf_geo": 0.15,
}

per_query_results = []
valid_predictions = {name: {} for name in geo_bonuses}

started = perf_counter()

for row_number, query in tqdm(
    enumerate(valid_queries.itertuples(index=False)),
    total=len(valid_queries),
):
    similarities = (X_valid[row_number] @ X_items_T).tocsr()
    similarities.sum_duplicates()
    similarities.eliminate_zeros()

    candidate_indices = similarities.indices
    text_scores = similarities.data

    # проверяем географию только среди найденных объявлений
    same_location = (
        corpus_locations[candidate_indices]
        == query.search_location_id
    )

    targets = valid_targets[query.val_id]

    for name, bonus in geo_bonuses.items():
        scores = text_scores + np.float32(bonus) * same_location

        k = min(50, len(scores))

        if k == 0:
            prediction = []
        else:
            if len(scores) > k:
                selected = np.argpartition(scores, -k)[-k:]
            else:
                selected = np.arange(len(scores))

            selected = selected[
                np.argsort(-scores[selected], kind="stable")
            ]

            prediction = corpus_ids[
                candidate_indices[selected]
            ].tolist()

        valid_predictions[name][query.val_id] = prediction
        hits = len(set(prediction) & targets)

        per_query_results.append({
            "val_id": query.val_id,
            "method": name,
            "recall_at_50": hits / len(targets),
            "hits": hits,
            "n_positive": len(targets),
            "n_returned": len(prediction),
        })

results = pd.DataFrame(per_query_results)

summary = (
    results.groupby("method")
    .agg(
        recall_at_50=("recall_at_50", "mean"),
        queries_with_hit=("hits", lambda x: (x > 0).mean()),
        mean_candidates=("n_returned", "mean"),
    )
    .sort_values("recall_at_50", ascending=False)
)

display(summary)
print(f"Время поиска и оценки: {perf_counter() - started:.1f} сек.")

Подготовка матрицы: 0.6 сек.


  0%|          | 0/2000 [00:00<?, ?it/s]

,recall_at_50,queries_with_hit,mean_candidates
method,,,
title_char_tfidf_geo,0.325929,0.3340,50.0
title_char_tfidf,0.195704,0.2005,50.0


Время поиска и оценки: 34.5 сек.


## 4. Эксперимент: объединение двух текстовых поисков

### 4.1. Словесный TF-IDF

Добавляем индекс по словам и парам соседних слов. Документ включает заголовок, первые 800 символов параметров и первые 2 000 символов описания. Заголовок повторяем трижды для увеличения его вклада. Ограничение длины уменьшает затраты памяти и времени; итоговый словарь содержит 200 тысяч признаков.


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from time import perf_counter

started = perf_counter()

text_columns = [
    "item_id",
    "item_title_raw",
    "item_infm_params_text",
    "item_description_raw",
]

# сохраняем тот же порядок объявлений, что и в символьном индексе
text_corpus = (
    pd.concat(
        [items[text_columns], train[text_columns]],
        ignore_index=True,
    )
    .drop_duplicates("item_id", keep="first")
    .reset_index(drop=True)
)

assert np.array_equal(text_corpus["item_id"].to_numpy(), corpus_ids)


def prepare_document(row):
    title, params, description = row

    title = "" if pd.isna(title) else str(title)
    params = "" if pd.isna(params) else str(params)[:800]
    description = (
        "" if pd.isna(description) else str(description)[:2000]
    )

    # повторение заголовка увеличивает его влияние в текстовом поиске
    text = f"{title} {title} {title} {params} {description}"
    return normalize_query(text)


word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.98,
    max_features=200_000,
    sublinear_tf=True,
    norm="l2",
    dtype=np.float32,
)

document_rows = text_corpus[
    [
        "item_title_raw",
        "item_infm_params_text",
        "item_description_raw",
    ]
].itertuples(index=False, name=None)

X_word = word_vectorizer.fit_transform(
    map(prepare_document, document_rows)
)

Q_word_valid = word_vectorizer.transform(valid_queries["query_norm"])
X_word_T = X_word.T.tocsr()

print(f"Размер нового индекса: {X_word.shape}")
print(f"Время построения: {perf_counter() - started:.1f} сек.")

del document_rows, text_corpus

Размер нового индекса: (515895, 200000)
Время построения: 387.3 сек.


### 4.2. Смешивание оценок

Проверяем веса словесного поиска 0, 0.5 и 1 и несколько географических бонусов. Перед смешиванием делим оценки каждого поиска на его максимум для данного запроса. Исходный символьный baseline оставляем отдельным контролем.

**Лучший вариант:** равные веса поисков и бонус локации 0.60, локальный Recall@50 - **0.627337**.


In [11]:
from tqdm.auto import tqdm

# сохраняем исходный вариант для честного сравнения
configs = [
    {
        "name": "original_char_geo",
        "word_weight": 0.0,
        "geo_bonus": 0.15,
        "normalize": False,
    }
]

for word_weight in (0.0, 0.5, 1.0):
    for geo_bonus in (0.15, 0.35, 0.60):
        configs.append({
            "name": f"word_{word_weight}_geo_{geo_bonus}",
            "word_weight": word_weight,
            "geo_bonus": geo_bonus,
            "normalize": True,
        })


def sparse_scores(query_matrix, row_number, index_transposed):
    result = (query_matrix[row_number] @ index_transposed).tocsr()
    result.sum_duplicates()
    result.eliminate_zeros()
    return result


def prepare_candidates(char_result, word_result, location):
    indices = np.union1d(char_result.indices, word_result.indices)

    char_scores = np.zeros(len(indices), dtype=np.float32)
    word_scores = np.zeros(len(indices), dtype=np.float32)

    char_scores[
        np.searchsorted(indices, char_result.indices)
    ] = char_result.data

    word_scores[
        np.searchsorted(indices, word_result.indices)
    ] = word_result.data

    same_location = corpus_locations[indices] == location
    return indices, char_scores, word_scores, same_location


def select_top50(prepared, config):
    indices, char_scores, word_scores, same_location = prepared

    if config["normalize"]:
        char_scores = char_scores / max(float(char_scores.max(initial=0)), 1e-9)
        word_scores = word_scores / max(float(word_scores.max(initial=0)), 1e-9)

    weight = config["word_weight"]
    text_scores = (1 - weight) * char_scores + weight * word_scores

    # география не должна самостоятельно создавать кандидатов с нулевым текстовым сходством выбранного метода
    active = np.flatnonzero(text_scores > 0)
    if len(active) == 0:
        return []

    scores = (
        text_scores[active]
        + config["geo_bonus"] * same_location[active]
    )

    k = min(50, len(active))
    selected = (
        np.argpartition(scores, -k)[-k:]
        if len(active) > k
        else np.arange(len(active))
    )
    selected = selected[np.argsort(-scores[selected], kind="stable")]

    return corpus_ids[indices[active[selected]]].tolist()


experiment_rows = []
started = perf_counter()

for i, query in tqdm(
    enumerate(valid_queries.itertuples(index=False)),
    total=len(valid_queries),
):
    char_result = sparse_scores(X_valid, i, X_items_T)
    word_result = sparse_scores(Q_word_valid, i, X_word_T)

    prepared = prepare_candidates(
        char_result, word_result, query.search_location_id
    )
    targets = valid_targets[query.val_id]

    for config in configs:
        prediction = select_top50(prepared, config)
        hits = len(set(prediction) & targets)

        experiment_rows.append({
            "val_id": query.val_id,
            "method": config["name"],
            "recall_at_50": hits / len(targets),
        })

experiment_results = pd.DataFrame(experiment_rows)

comparison = (
    experiment_results.groupby("method")["recall_at_50"]
    .mean()
    .sort_values(ascending=False)
    .to_frame()
)

display(comparison)

best_name = comparison.index[0]
best_config = next(
    config.copy() for config in configs
    if config["name"] == best_name
)

print(f"Выбранный вариант: {best_config}")
print(f"Время проверки: {perf_counter() - started:.1f} сек.")

  0%|          | 0/2000 [00:00<?, ?it/s]

,recall_at_50
method,
word_0.5_geo_0.6,0.627337
word_1.0_geo_0.6,0.578887
word_0.5_geo_0.35,0.541554
word_0.0_geo_0.6,0.540138
word_1.0_geo_0.35,0.490413
word_0.0_geo_0.35,0.442421
word_0.5_geo_0.15,0.373004
original_char_geo,0.325929
word_1.0_geo_0.15,0.314263


Выбранный вариант: {'name': 'word_0.5_geo_0.6', 'word_weight': 0.5, 'geo_bonus': 0.6, 'normalize': True}
Время проверки: 124.0 сек.


### 4.3. Первая отправка

Применяем выбранные настройки к запросам бенчмарка. Используем часть готовых индексов, соответствующую `benchmark_items`. Проверяем состав ответа и записываем answer.csv в UTF-8: два столбца, без индекса, до 50 уникальных `item_id` через пробел.


In [13]:
from pathlib import Path
import json

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

n_benchmark_items = len(items)

# в корпусе benchmark_items идут первыми
assert np.array_equal(
    corpus_ids[:n_benchmark_items],
    items["item_id"].to_numpy(),
)

benchmark_texts = queries["search_query"].map(normalize_query)

Q_char_benchmark = vectorizer.transform(benchmark_texts)
Q_word_benchmark = word_vectorizer.transform(benchmark_texts)

char_benchmark_T = X_items[:n_benchmark_items].T.tocsr()
word_benchmark_T = X_word[:n_benchmark_items].T.tocsr()

predictions = []

for i, query in tqdm(
    enumerate(queries.itertuples(index=False)),
    total=len(queries),
):
    char_result = sparse_scores(
        Q_char_benchmark, i, char_benchmark_T
    )
    word_result = sparse_scores(
        Q_word_benchmark, i, word_benchmark_T
    )

    prepared = prepare_candidates(
        char_result, word_result, query.search_location_id
    )
    predictions.append(select_top50(prepared, best_config))

answer = pd.DataFrame({
    "query_id": queries["query_id"].to_numpy(),
    "answer": [" ".join(ids) for ids in predictions],
})

# проверки формата и принадлежности идентификаторов корпусу
allowed_ids = set(items["item_id"])

assert list(answer.columns) == ["query_id", "answer"]
assert len(answer) == len(queries)
assert answer["query_id"].is_unique
assert set(answer["query_id"]) == set(queries["query_id"])
assert answer["query_id"].map(lambda value: isinstance(value, str) and len(value) == 16).all()

for ids in predictions:
    assert len(ids) <= 50
    assert len(ids) == len(set(ids))
    assert set(ids).issubset(allowed_ids)
    assert all(re.fullmatch(r"[0-9a-f]{16}", value) for value in ids)

answer_path = OUTPUT_DIR / "answer.csv"
answer.to_csv(answer_path, index=False, encoding="utf-8")

answer_path = OUTPUT_DIR / "answer.csv"
answer.to_csv(answer_path, index=False, encoding="utf-8")

print("Файл сохранён:", answer_path)
print("Строк:", len(answer))
print(
    "Запросов с менее чем 50 кандидатами:",
    sum(len(ids) < 50 for ids in predictions),
)
display(answer.head(3))

  0%|          | 0/2452 [00:00<?, ?it/s]

Файл сохранён: /kaggle/working/answer.csv
Строк: 2452
Запросов с менее чем 50 кандидатами: 0


,query_id,answer
0,70DfDUpwjxB4lzFd,d722bcda1a555091 fa01652ca5f57d0f 603623b4bd9e...
1,JTrdTaZJvSiLPkXj,16ea26ff0aa0450a 8703b2a0442c5d25 cbeccbecb1fb...
2,LZCZNoVG4AFUkVRJ,3f89b8062dc85f1c d8fce513e4f000a7 dab52187b450...


**Результат отправки 1:** Recall@50 на Stepik - **0.698514**.


## 5. Эксперимент: уточнение весов текста и географии

Проверяем веса словесного поиска 0.35, 0.50 и 0.65 и более сильные географические бонусы. При равных оценках фиксируем выбор по индексу объявления, поэтому контрольный результат немного отличается от предыдущего запуска.

**Лучший вариант:** вес словесного поиска **0.65**, географический бонус **0.80**, локальный Recall@50 - **0.649796**. По сравнению с контролем улучшились 72 запроса, ухудшились 27.


In [14]:
geo_configs = []

for word_weight in (0.35, 0.50, 0.65):
    for geo_bonus in (0.60, 0.80, 1.00, 1.30):
        geo_configs.append({
            "name": f"word_{word_weight:.2f}_geo_{geo_bonus:.2f}",
            "word_weight": word_weight,
            "geo_bonus": geo_bonus,
            "normalize": True,
        })


def get_sparse_similarity(query_matrix, row_number, index_transposed):
    similarity = (
        query_matrix[row_number] @ index_transposed
    ).tocsr()

    similarity.sum_duplicates()
    similarity.eliminate_zeros()

    return similarity


def collect_candidate_scores(char_result, word_result, location):
    indices = np.union1d(char_result.indices, word_result.indices)

    char_scores = np.zeros(len(indices), dtype=np.float32)
    word_scores = np.zeros(len(indices), dtype=np.float32)

    char_scores[
        np.searchsorted(indices, char_result.indices)
    ] = char_result.data

    word_scores[
        np.searchsorted(indices, word_result.indices)
    ] = word_result.data

    char_max = float(char_scores.max()) if len(indices) else 0.0
    word_max = float(word_scores.max()) if len(indices) else 0.0

    if char_max > 0:
        char_scores /= char_max

    if word_max > 0:
        word_scores /= word_max

    same_location = corpus_locations[indices] == location

    return indices, char_scores, word_scores, same_location


def choose_candidates(prepared, config, k=50):
    indices, char_scores, word_scores, same_location = prepared

    weight = config["word_weight"]
    text_scores = (
        (1 - weight) * char_scores
        + weight * word_scores
    )

    active = np.flatnonzero(text_scores > 0)

    if len(active) == 0:
        return []

    scores = (
        text_scores[active]
        + config["geo_bonus"] * same_location[active]
    )

    if len(active) > k:
        threshold = np.partition(scores, len(scores) - k)[
            len(scores) - k
        ]

        better = np.flatnonzero(scores > threshold)
        tied = np.flatnonzero(scores == threshold)

        remaining = k - len(better)
        selected = np.concatenate([better, tied[:remaining]])
    else:
        selected = np.arange(len(active))

    order = np.lexsort((
        indices[active[selected]],
        -scores[selected],
    ))
    selected = selected[order]

    return corpus_ids[indices[active[selected]]].tolist()


geo_rows = []
started = perf_counter()

for row_number, query in tqdm(
    enumerate(valid_queries.itertuples(index=False)),
    total=len(valid_queries),
):
    char_result = get_sparse_similarity(
        X_valid, row_number, X_items_T
    )
    word_result = get_sparse_similarity(
        Q_word_valid, row_number, X_word_T
    )

    prepared = collect_candidate_scores(
        char_result,
        word_result,
        query.search_location_id,
    )

    targets = valid_targets[query.val_id]

    for config in geo_configs:
        prediction = choose_candidates(prepared, config)
        hits = len(set(prediction) & targets)

        geo_rows.append({
            "val_id": query.val_id,
            "method": config["name"],
            "recall_at_50": hits / len(targets),
        })


geo_results = pd.DataFrame(geo_rows)

geo_per_query = geo_results.pivot(
    index="val_id",
    columns="method",
    values="recall_at_50",
)

reference_name = "word_0.50_geo_0.60"
reference_scores = geo_per_query[reference_name]

geo_comparison = pd.DataFrame({
    "recall_at_50": geo_per_query.mean(),
    "delta_vs_previous": (
        geo_per_query.sub(reference_scores, axis=0).mean()
    ),
    "queries_improved": (
        geo_per_query.gt(reference_scores, axis=0).sum()
    ),
    "queries_worsened": (
        geo_per_query.lt(reference_scores, axis=0).sum()
    ),
}).sort_values("recall_at_50", ascending=False)

display(geo_comparison)

geo_best_name = geo_comparison.index[0]
geo_best_config = next(
    config.copy()
    for config in geo_configs
    if config["name"] == geo_best_name
)

print(f"Лучшие настройки этого эксперимента: {geo_best_config}")
print(f"Время проверки: {perf_counter() - started:.1f} сек.")

  0%|          | 0/2000 [00:00<?, ?it/s]

,recall_at_50,delta_vs_previous,queries_improved,queries_worsened
method,,,,
word_0.65_geo_0.80,0.649796,0.021958,72,27
word_0.65_geo_1.00,0.648796,0.020958,74,31
word_0.65_geo_1.30,0.648796,0.020958,74,31
word_0.50_geo_0.80,0.640813,0.012975,34,8
word_0.65_geo_0.60,0.640571,0.012733,50,23
word_0.50_geo_1.00,0.639813,0.011975,38,14
word_0.50_geo_1.30,0.639813,0.011975,38,14
word_0.50_geo_0.60,0.627837,0.000000,0,0
word_0.35_geo_0.80,0.626154,-0.001683,39,43


Лучшие настройки этого эксперимента: {'name': 'word_0.65_geo_0.80', 'word_weight': 0.65, 'geo_bonus': 0.8, 'normalize': True}
Время проверки: 132.6 сек.


### 5.1. Вторая отправка

Формируем ответ с уточненными настройками. После записи читаем CSV обратно и проверяем, что сохраненная таблица совпадает с подготовленной.


In [15]:
# используем "победителя" последнего локального эксперимента
submission_config = geo_best_config.copy()
print("Настройки:", submission_config)

# проверяем соответствие порядка объявлений двум индексам
n_benchmark_items = len(items)


tuned_predictions = []
started = perf_counter()

for row_number, query in tqdm(
    enumerate(queries.itertuples(index=False)),
    total=len(queries),
):
    char_result = get_sparse_similarity(
        Q_char_benchmark,
        row_number,
        char_benchmark_T,
    )
    word_result = get_sparse_similarity(
        Q_word_benchmark,
        row_number,
        word_benchmark_T,
    )

    prepared = collect_candidate_scores(
        char_result,
        word_result,
        query.search_location_id,
    )

    tuned_predictions.append(
        choose_candidates(prepared, submission_config)
    )

answer = pd.DataFrame({
    "query_id": queries["query_id"].to_numpy(),
    "answer": [" ".join(ids) for ids in tuned_predictions],
})

# проверяем состав запросов и формат ответа
allowed_ids = set(items["item_id"])

assert list(answer.columns) == ["query_id", "answer"]
assert answer["query_id"].is_unique
assert answer["query_id"].tolist() == queries["query_id"].tolist()


for ids in tuned_predictions:
    assert len(ids) <= 50
    assert len(ids) == len(set(ids))
    assert set(ids).issubset(allowed_ids)
    assert all(
        re.fullmatch(r"[0-9a-f]{16}", item_id)
        for item_id in ids
    )

answer_path = OUTPUT_DIR / "answer.csv"
answer.to_csv(answer_path, index=False, encoding="utf-8")

# дополнительно проверим записанный файл
saved_answer = pd.read_csv(
    answer_path,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8",
)
pd.testing.assert_frame_equal(saved_answer, answer)

print(f"Проверки пройдены: {answer_path}")
print(f"Строк: {len(answer)}")
print(f"Запросов с менее чем 50 кандидатами: {sum(len(ids) < 50 for ids in tuned_predictions)}")
print(f"Время: {perf_counter() - started:.1f} сек.")
display(answer.head(3))

Настройки: {'name': 'word_0.65_geo_0.80', 'word_weight': 0.65, 'geo_bonus': 0.8, 'normalize': True}


  0%|          | 0/2452 [00:00<?, ?it/s]

Проверки пройдены: /kaggle/working/answer.csv
Строк: 2452
Запросов с менее чем 50 кандидатами: 0
Время: 35.7 сек.


,query_id,answer
0,70DfDUpwjxB4lzFd,b92ee8f432cec2d1 d722bcda1a555091 fa01652ca5f5...
1,JTrdTaZJvSiLPkXj,16ea26ff0aa0450a 8703b2a0442c5d25 cbeccbecb1fb...
2,LZCZNoVG4AFUkVRJ,3f89b8062dc85f1c d8fce513e4f000a7 dab52187b450...


**Результат отправки 2:** Recall@50 на Stepik - **0.709335**. Уточнение весов улучшило и локальный результат, и оценку платформы.


## 6. Эксперимент: CatBoost на текстовых кандидатах

### 6.1. Выбор обучающих запросов

Выбираем 8 000 текстов из обучающей части и по одному случайному контексту для каждого. Они не пересекаются с текстами валидации. Для выбранных контекстов получено **8 446 положительных пар**. Подвыборка ограничивает время построения кандидатов и обучения.


In [18]:
# fit_pairs уже исключает все тексты из valid_texts

N_FIT_QUERIES = 8_000

fit_contexts = (
    fit_pairs[query_features + ["query_norm"]]
    .drop_duplicates()
)

fit_available_texts = np.array(
    sorted(fit_contexts["query_norm"].unique())
)

fit_rng = np.random.default_rng(42)

selected_fit_texts = set(
    fit_rng.choice(
        fit_available_texts,
        size=min(N_FIT_QUERIES, len(fit_available_texts)),
        replace=False,
    )
)

# один случайный контекст на выбранный текст
ranker_queries = (
    fit_contexts[
        fit_contexts["query_norm"].isin(selected_fit_texts)
    ]
    .sample(frac=1, random_state=42)
    .drop_duplicates("query_norm")
    .reset_index(drop=True)
)

ranker_queries.insert(
    0,
    "fit_id",
    np.arange(len(ranker_queries), dtype=np.int64),
)

ranker_positive_rows = ranker_queries[
    ["fit_id"] + query_features
].merge(
    fit_pairs[query_features + ["item_id"]],
    on=query_features,
    how="inner",
    validate="one_to_many",
)

ranker_targets = (
    ranker_positive_rows.groupby("fit_id")["item_id"]
    .agg(set)
    .to_dict()
)

assert not set(ranker_queries["query_norm"]) & valid_texts
assert len(ranker_targets) == len(ranker_queries)

print(f"Обучающих запросов: {len(ranker_queries)}")
print(f"Известных положительных пар: {sum(len(ids) for ids in ranker_targets.values())}")
print("Пересечений текстов с валидацией нет")

Обучающих запросов: 8000
Известных положительных пар: 8446
Пересечений текстов с валидацией нет


### 6.2. Обучаемый отбор кандидатов

Вместо выбора по одной формуле обучаем CatBoostRanker на результатах двух поисков и признаках объявлений. Цель этого этапа - отобрать 50 кандидатов из более широкого набора.


#### Признаки и генерация кандидатов

Объединяем до 200 кандидатов поиска с географией и до 100 - без географического бонуса. Формируем 20 признаков: исходные и нормированные текстовые оценки, ранги, совпадение локации, длины текстов, рейтинг, число отзывов, цену и флаги связи. Пропуски числовых характеристик кодируем отдельно.


In [19]:
from catboost import CatBoostRanker

# cобираем компактные признаки в том же порядке, что и поисковые индексы
metadata_columns = [
    "item_id",
    "item_rating",
    "item_rating_reviews_count",
    "item_price",
    "item_is_phone_hidden",
    "item_is_message_forbidden",
]

ranker_metadata = (
    pd.concat(
        [items[metadata_columns], train[metadata_columns]],
        ignore_index=True,
    )
    .drop_duplicates("item_id", keep="first")
    .reset_index(drop=True)
)

assert np.array_equal(
    ranker_metadata["item_id"].to_numpy(),
    corpus_ids,
)

rating_values = pd.to_numeric(
    ranker_metadata["item_rating"], errors="coerce"
)
review_values = pd.to_numeric(
    ranker_metadata["item_rating_reviews_count"], errors="coerce"
)
price_values = pd.to_numeric(
    ranker_metadata["item_price"], errors="coerce"
)

# пропуски кодируем отдельно, чтобы не путать их с настоящими нулями
item_numeric_features = np.column_stack([
    rating_values.fillna(-1).to_numpy(),
    rating_values.isna().to_numpy(),
    np.log1p(review_values.fillna(0).clip(lower=0).to_numpy()),
    review_values.isna().to_numpy(),
    np.log1p(price_values.fillna(0).clip(lower=0).to_numpy()),
    price_values.isna().to_numpy(),
    ranker_metadata["item_is_phone_hidden"].to_numpy(),
    ranker_metadata["item_is_message_forbidden"].to_numpy(),
]).astype(np.float32)

# длину заголовка считаем без создания дополнительных больших текстов
item_title_lengths = (
    corpus["item_title_raw"]
    .fillna("")
    .str.len()
    .to_numpy(dtype=np.float32)
)

del ranker_metadata, rating_values, review_values, price_values

feature_names = [
    "char_score",
    "word_score",
    "char_score_normalized",
    "word_score_normalized",
    "text_mix",
    "text_mix_with_geo",
    "same_location",
    "char_rank_in_pool",
    "word_rank_in_pool",
    "query_chars",
    "query_words",
    "title_chars",
    "rating",
    "rating_missing",
    "log_reviews",
    "reviews_missing",
    "log_price",
    "price_missing",
    "phone_hidden",
    "messages_forbidden",
]


def stable_top_positions(scores, k):
    active = np.flatnonzero(scores > 0)

    if len(active) <= k:
        return active

    active_scores = scores[active]
    threshold = np.partition(
        active_scores, len(active_scores) - k
    )[len(active_scores) - k]

    better = active[active_scores > threshold]
    tied = active[active_scores == threshold]

    return np.concatenate([
        better,
        tied[:k - len(better)],
    ])


def ranks_in_pool(scores):
    order = np.argsort(-scores, kind="stable")
    ranks = np.empty(len(scores), dtype=np.float32)
    ranks[order] = np.arange(1, len(scores) + 1)
    return ranks


def make_ranker_candidates(char_result, word_result, location, query_text):
    indices = np.union1d(char_result.indices, word_result.indices)

    if len(indices) == 0:
        return (
            np.empty(0, dtype=np.int64),
            np.empty((0, len(feature_names)), dtype=np.float32),
        )

    char_raw = np.zeros(len(indices), dtype=np.float32)
    word_raw = np.zeros(len(indices), dtype=np.float32)

    char_raw[
        np.searchsorted(indices, char_result.indices)
    ] = char_result.data

    word_raw[
        np.searchsorted(indices, word_result.indices)
    ] = word_result.data

    char_norm = char_raw / max(float(char_raw.max()), 1e-9)
    word_norm = word_raw / max(float(word_raw.max()), 1e-9)

    same_location = corpus_locations[indices] == location

    text_mix = 0.35 * char_norm + 0.65 * word_norm
    geo_mix = text_mix + 0.80 * same_location

    # не допускаем кандидатов только за счёт географии
    geo_mix[text_mix <= 0] = 0

    # до 200 кандидатов с географией и до 100 без неё, после объединения повторов нет
    selected = np.union1d(
        stable_top_positions(geo_mix, 200),
        stable_top_positions(text_mix, 100),
    )

    candidate_indices = indices[selected]
    n_candidates = len(selected)

    features = np.column_stack([
        char_raw[selected],
        word_raw[selected],
        char_norm[selected],
        word_norm[selected],
        text_mix[selected],
        geo_mix[selected],
        same_location[selected],
        ranks_in_pool(char_raw[selected]),
        ranks_in_pool(word_raw[selected]),
        np.full(n_candidates, len(query_text)),
        np.full(n_candidates, len(query_text.split())),
        item_title_lengths[candidate_indices],
        item_numeric_features[candidate_indices],
    ]).astype(np.float32)

    return candidate_indices, features

### 6.3. Таблицы для обучения и проверки

Размечаем найденных кандидатов по известным выборам пользователей; остальные считаются условно отрицательными. Правильные объявления искусственно не добавляем. При обучении пропускаем группы без различия классов, но в валидации учитываем все запросы.

Для обучения осталось **6 311 групп**. Recall расширенного набора на валидации - **0.776104**: это верхняя граница качества отбора из этих кандидатов.


In [20]:
def build_ranker_dataset(query_frame, targets, id_column, training=False):
    query_char = vectorizer.transform(query_frame["query_norm"])
    query_word = word_vectorizer.transform(query_frame["query_norm"])

    feature_parts = []
    label_parts = []
    group_parts = []
    candidate_parts = []

    pool_recalls = []
    skipped_groups = 0

    for row_number, query in tqdm(
        enumerate(query_frame.itertuples(index=False)),
        total=len(query_frame),
    ):
        query_id = getattr(query, id_column)

        char_result = get_sparse_similarity(
            query_char, row_number, X_items_T
        )
        word_result = get_sparse_similarity(
            query_word, row_number, X_word_T
        )

        candidate_indices, features = make_ranker_candidates(
            char_result,
            word_result,
            query.search_location_id,
            query.query_norm,
        )

        positive_ids = targets[query_id]

        labels = np.array([
            item_id in positive_ids
            for item_id in corpus_ids[candidate_indices]
        ], dtype=np.int8)

        pool_recalls.append(
            float(labels.sum()) / len(positive_ids)
        )

        # для обучения нужны кандидаты обоих классов
        if training and (
            len(labels) == 0
            or labels.sum() == 0
            or labels.sum() == len(labels)
        ):
            skipped_groups += 1
            continue

        if len(labels) == 0:
            continue

        feature_parts.append(features)
        label_parts.append(labels)
        group_parts.append(
            np.full(len(labels), query_id, dtype=np.int64)
        )
        candidate_parts.append(candidate_indices)

    if not feature_parts:
        raise ValueError("Не удалось собрать группы кандидатов")

    dataset = {
        "X": np.concatenate(feature_parts),
        "y": np.concatenate(label_parts),
        "group": np.concatenate(group_parts),
        "candidate_index": np.concatenate(candidate_parts),
    }

    print(f"Пар запрос–кандидат: {len(dataset["y"])}")
    print(f"Групп в таблице: {len(np.unique(dataset["group"]))}")
    print(f"Пропущено обучающих групп: {skipped_groups}")
    print(f"Recall расширенного набора по всем исходным запросам: {round(float(np.mean(pool_recalls)), 6)}")

    return dataset


started = perf_counter()

print("Обучающая выборка:")
rank_train = build_ranker_dataset(
    ranker_queries,
    ranker_targets,
    id_column="fit_id",
    training=True,
)

print("\nВалидационная выборка:")
rank_valid = build_ranker_dataset(
    valid_queries,
    valid_targets,
    id_column="val_id",
    training=False,
)

print(f"\nВремя подготовки: {perf_counter() - started:.1f} сек.")

Обучающая выборка:


  0%|          | 0/8000 [00:00<?, ?it/s]

Пар запрос–кандидат: 1732029
Групп в таблице: 6311
Пропущено обучающих групп: 1689
Recall расширенного набора по всем исходным запросам: 0.783392

Валидационная выборка:


  0%|          | 0/2000 [00:00<?, ?it/s]

Пар запрос–кандидат: 537323
Групп в таблице: 2000
Пропущено обучающих групп: 0
Recall расширенного набора по всем исходным запросам: 0.776104

Время подготовки: 415.6 сек.


### 6.4. CatBoostRanker

Обучаем модель с функцией потерь YetiRank. Для ранней остановки используем NDCG@50, а итог сравниваем по целевой Recall@50. В знаменателе остаются все известные позитивы, в том числе не найденные при генерации кандидатов.

**Результат:** модель использует 376 деревьев и дает **0.652462** против **0.649796** у текстового поиска. Прирост небольшой: 54 запроса улучшились, 47 ухудшились.


In [22]:
from catboost import Pool

# pool хранит признаки, разметку и принадлежность строк к запросам
train_pool = Pool(
    data=rank_train["X"],
    label=rank_train["y"],
    group_id=rank_train["group"],
    feature_names=feature_names,
)

valid_pool = Pool(
    data=rank_valid["X"],
    label=rank_valid["y"],
    group_id=rank_valid["group"],
    feature_names=feature_names,
)

ranker_model = CatBoostRanker(
    iterations=600,
    depth=6,
    learning_rate=0.05,
    loss_function="YetiRank",
    eval_metric="NDCG:top=50",
    task_type="GPU",
    devices="0",
    random_seed=42,
    verbose=100,
    allow_writing_files=False,
)

started = perf_counter()
print("Начинаем обучение CatBoost", flush=True)

ranker_model.fit(
    train_pool,
    eval_set=valid_pool,
    early_stopping_rounds=80,
    use_best_model=True,
)

print(f"Время обучения: {perf_counter() - started:.1f} сек.")

ranker_valid_scores = ranker_model.predict(valid_pool)


def evaluate_candidate_scores(dataset, scores, targets, method):
    groups = dataset["group"]

    # находим границы групп: строки одного запроса идут подряд
    boundaries = np.flatnonzero(
        np.r_[True, groups[1:] != groups[:-1], True]
    )

    # запросы без найденных кандидатов учитываются с нулевым recall
    query_recalls = {query_id: 0.0 for query_id in targets}

    for start, end in zip(boundaries[:-1], boundaries[1:]):
        query_id = int(groups[start])
        candidate_indices = dataset["candidate_index"][start:end]
        group_scores = scores[start:end]

        # при равных оценках порядок задаём индексом объявления
        selected = np.lexsort((
            candidate_indices,
            -group_scores,
        ))[:50]

        hits = dataset["y"][start:end][selected].sum()
        query_recalls[query_id] = (
            float(hits) / len(targets[query_id])
        )

    return pd.Series(query_recalls, name=method)


# сравниваем модель с текущим поиском на тех же запросах
baseline_valid_scores = rank_valid["X"][:, feature_names.index("text_mix_with_geo")]

ranker_recall_by_query = evaluate_candidate_scores(
    rank_valid,
    ranker_valid_scores,
    valid_targets,
    "catboost",
)

baseline_recall_by_query = evaluate_candidate_scores(
    rank_valid,
    baseline_valid_scores,
    valid_targets,
    "lexical",
)

ranker_comparison = pd.DataFrame({
    "recall_at_50": [
        baseline_recall_by_query.mean(),
        ranker_recall_by_query.mean(),
    ],
}, index=["lexical", "catboost"])

display(ranker_comparison)

difference = ranker_recall_by_query - baseline_recall_by_query

print(f"Запросов улучшилось: {int((difference > 0).sum())}")
print(f"Запросов ухудшилось: {int((difference < 0).sum())}")
print(f"Изменение Recall@50: {round(float(difference.mean()), 6)}")
print(f"Использовано деревьев: {ranker_model.tree_count_}")

Начинаем обучение CatBoost


Default metric period is 5 because PFound, NDCG is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=50;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.4551166	best: 0.4551166 (0)	total: 411ms	remaining: 4m 6s
100:	test: 0.5706197	best: 0.5709736 (99)	total: 8.95s	remaining: 44.2s
200:	test: 0.5770526	best: 0.5770537 (199)	total: 17.4s	remaining: 34.5s
300:	test: 0.5787380	best: 0.5787380 (300)	total: 25.8s	remaining: 25.7s
400:	test: 0.5796029	best: 0.5799090 (375)	total: 34.3s	remaining: 17s
bestTest = 0.5799090144
bestIteration = 375
Shrink model to first 376 iterations.
Время обучения: 40.9 сек.


,recall_at_50
lexical,0.649796
catboost,0.652462


Запросов улучшилось: 54
Запросов ухудшилось: 47
Изменение Recall@50: 0.002667
Использовано деревьев: 376


### 6.5. Объединение поиска и CatBoost

Поскольку оценки модели и поиска имеют разные масштабы, объединяем их через взвешенный RRF: вклад списка равен `вес / (60 + ранг)`, ранги начинаются с 1. Проверяем несколько весов CatBoost.

**Результат:** вес CatBoost **0.75** дает Recall@50 **0.657046**, выше обоих отдельных методов. Улучшились 48 запросов, ухудшились 32 относительно текстового поиска.


In [23]:
# alpha=0 - только поиск, alpha=1 - только CatBoost
blend_alphas = (0.0, 0.25, 0.50, 0.75, 1.0)
rrf_constant = 60.0

groups = rank_valid["group"]

group_boundaries = np.flatnonzero(
    np.r_[True, groups[1:] != groups[:-1], True]
)

lexical_rrf = np.zeros(len(groups), dtype=np.float32)
catboost_rrf = np.zeros(len(groups), dtype=np.float32)

for start, end in zip(
    group_boundaries[:-1],
    group_boundaries[1:],
):
    candidate_indices = rank_valid["candidate_index"][start:end]

    # одинаковые оценки разрешаем по индексу объявления
    lexical_order = np.lexsort((
        candidate_indices,
        -baseline_valid_scores[start:end],
    ))
    catboost_order = np.lexsort((
        candidate_indices,
        -ranker_valid_scores[start:end],
    ))

    lexical_ranks = np.empty(end - start, dtype=np.float32)
    catboost_ranks = np.empty(end - start, dtype=np.float32)

    lexical_ranks[lexical_order] = np.arange(1, end - start + 1)
    catboost_ranks[catboost_order] = np.arange(1, end - start + 1)

    lexical_rrf[start:end] = 1.0 / (
        rrf_constant + lexical_ranks
    )
    catboost_rrf[start:end] = 1.0 / (
        rrf_constant + catboost_ranks
    )

blend_rows = []
blend_recalls = {}

for alpha in blend_alphas:
    blended_scores = (
        (1 - alpha) * lexical_rrf
        + alpha * catboost_rrf
    )

    recall_by_query = evaluate_candidate_scores(
        rank_valid,
        blended_scores,
        valid_targets,
        method=f"catboost_weight_{alpha:.2f}",
    )

    blend_recalls[alpha] = recall_by_query
    delta = recall_by_query - baseline_recall_by_query

    blend_rows.append({
        "catboost_weight": alpha,
        "recall_at_50": recall_by_query.mean(),
        "delta_vs_lexical": delta.mean(),
        "queries_improved": int((delta > 0).sum()),
        "queries_worsened": int((delta < 0).sum()),
    })

blend_comparison = (
    pd.DataFrame(blend_rows)
    .sort_values(
        ["recall_at_50", "catboost_weight"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(blend_comparison)

best_blend_alpha = float(
    blend_comparison.iloc[0]["catboost_weight"]
)

print(f"Лучший вес CatBoost: {best_blend_alpha}")

,catboost_weight,recall_at_50,delta_vs_lexical,queries_improved,queries_worsened
0,0.75,0.657046,0.007250,48,32
1,0.50,0.655546,0.005750,33,21
2,0.25,0.652546,0.002750,14,8
3,1.00,0.652462,0.002667,54,47
4,0.00,0.649796,0.000000,0,0


Лучший вес CatBoost: 0.75


### 6.6. Третья отправка

Для запросов бенчмарка строим тот же набор текстовых кандидатов, применяем первый CatBoost и объединяем ранги с весами 0.25 для поиска и 0.75 для модели. Проверяем формат и записываем ответ.


In [24]:
blend_alpha = 0.75
blend_predictions = []

started = perf_counter()

for row_number, query in tqdm(
    enumerate(queries.itertuples(index=False)),
    total=len(queries),
):
    char_result = get_sparse_similarity(
        Q_char_benchmark, row_number, char_benchmark_T
    )
    word_result = get_sparse_similarity(
        Q_word_benchmark, row_number, word_benchmark_T
    )

    candidate_indices, features = make_ranker_candidates(
        char_result,
        word_result,
        query.search_location_id,
        normalize_query(query.search_query),
    )

    if len(candidate_indices) == 0:
        blend_predictions.append([])
        continue

    lexical_scores = features[
        :, feature_names.index("text_mix_with_geo")
    ]
    model_scores = ranker_model.predict(features)

    lexical_order = np.lexsort((
        candidate_indices,
        -lexical_scores,
    ))
    model_order = np.lexsort((
        candidate_indices,
        -model_scores,
    ))

    lexical_ranks = np.empty(len(features), dtype=np.float32)
    model_ranks = np.empty(len(features), dtype=np.float32)

    lexical_ranks[lexical_order] = np.arange(1, len(features) + 1)
    model_ranks[model_order] = np.arange(1, len(features) + 1)

    scores = (
        (1 - blend_alpha) / (60.0 + lexical_ranks)
        + blend_alpha / (60.0 + model_ranks)
    )

    selected = np.lexsort((
        candidate_indices,
        -scores,
    ))[:50]

    blend_predictions.append(
        corpus_ids[candidate_indices[selected]].tolist()
    )

answer = pd.DataFrame({
    "query_id": queries["query_id"].to_numpy(),
    "answer": [" ".join(ids) for ids in blend_predictions],
})

allowed_ids = set(items["item_id"])

assert list(answer.columns) == ["query_id", "answer"]
assert answer["query_id"].is_unique
assert answer["query_id"].tolist() == queries["query_id"].tolist()

for ids in blend_predictions:
    assert len(ids) <= 50
    assert len(ids) == len(set(ids))
    assert set(ids).issubset(allowed_ids)
    assert all(re.fullmatch(r"[0-9a-f]{16}", value) for value in ids)

answer_path = OUTPUT_DIR / "answer.csv"
answer.to_csv(answer_path, index=False, encoding="utf-8")

saved_answer = pd.read_csv(
    answer_path,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8",
)
pd.testing.assert_frame_equal(saved_answer, answer)

print(f"Файл проверен: {answer_path}")
print(f"Время: {perf_counter() - started:.1f} сек.")

  0%|          | 0/2452 [00:00<?, ?it/s]

Файл проверен: /kaggle/working/answer.csv
Время: 43.8 сек.


**Результат отправки 3:** Recall@50 на Stepik - **0.713304**.


## 7. Эксперимент: смысловой поиск с E5

### 7.1. Модель и оценка скорости

Используем готовую `intfloat/multilingual-e5-small` без дообучения. Текст объявления содержит заголовок, до 200 символов параметров и до 600 символов описания с префиксом `passage: `. Токенизация ограничена 160 токенами. Усредняем представления без padding и нормируем векторы.

Перед полным расчетом измеряем скорость на 2 048 объявлениях. В сохраненном запуске получено около **996 объявлений/с** на одной T4.


In [25]:
import torch
from transformers import AutoTokenizer, AutoModel

assert torch.cuda.is_available(), "Нужен включённый GPU."

embedding_device = torch.device(
    "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
)

embedding_model_name = "intfloat/multilingual-e5-small"

embedding_tokenizer = AutoTokenizer.from_pretrained(
    embedding_model_name
)
embedding_model = AutoModel.from_pretrained(
    embedding_model_name
).to(embedding_device).half().eval()

print("Устройство:", embedding_device)
print("GPU:", torch.cuda.get_device_name(embedding_device))

# короткий текст для смыслового представления: заголовок, начало параметров и начало описания
embedding_columns = [
    "item_id",
    "item_title_raw",
    "item_infm_params_text",
    "item_description_raw",
]

embedding_corpus = (
    pd.concat(
        [items[embedding_columns], train[embedding_columns]],
        ignore_index=True,
    )
    .drop_duplicates("item_id", keep="first")
    .reset_index(drop=True)
)

assert np.array_equal(
    embedding_corpus["item_id"].to_numpy(),
    corpus_ids,
)


def make_embedding_text(row):
    title, params, description = row

    title = "" if pd.isna(title) else str(title)
    params = "" if pd.isna(params) else str(params)[:200]
    description = (
        "" if pd.isna(description) else str(description)[:600]
    )

    return f"passage: {title}. {params}. {description}"


@torch.inference_mode()
def encode_text_batch(texts):
    tokens = embedding_tokenizer(
        texts,
        max_length=160,
        padding=True,
        truncation=True,
        return_tensors="pt",
    ).to(embedding_device)

    outputs = embedding_model(**tokens)

    # среднее по токенам без учёта padding (усреднение считаем в float32 для устойчивости)
    hidden = outputs.last_hidden_state.float()
    mask = tokens["attention_mask"].unsqueeze(-1).bool()

    pooled = hidden.masked_fill(~mask, 0.0).sum(dim=1)
    pooled = pooled / mask.sum(dim=1).clamp(min=1)

    pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)

    return pooled.cpu().numpy()


pilot_frame = embedding_corpus.sample(
    n=min(2048, len(embedding_corpus)),
    random_state=42,
)

pilot_texts = list(map(
    make_embedding_text,
    pilot_frame[
        [
            "item_title_raw",
            "item_infm_params_text",
            "item_description_raw",
        ]
    ].itertuples(index=False, name=None),
))

EMBEDDING_BATCH_SIZE = 128

# прогрев отдельно от замера
_ = encode_text_batch(pilot_texts[:EMBEDDING_BATCH_SIZE])
torch.cuda.synchronize(embedding_device)

started = perf_counter()

for start in tqdm(
    range(0, len(pilot_texts), EMBEDDING_BATCH_SIZE)
):
    _ = encode_text_batch(
        pilot_texts[start:start + EMBEDDING_BATCH_SIZE]
    )

torch.cuda.synchronize(embedding_device)

pilot_seconds = perf_counter() - started
documents_per_second = len(pilot_texts) / pilot_seconds
estimated_minutes = (
    len(embedding_corpus) / documents_per_second / 60
)

print(f"Скорость: {documents_per_second:.1f} объявлений/сек.")
print(f"Приблизительное время кодирования всего корпуса: {estimated_minutes:.1f} мин.")

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Устройство: cuda:1
GPU: Tesla T4


  0%|          | 0/16 [00:00<?, ?it/s]

Скорость: 995.8 объявлений/сек.
Приблизительное время кодирования всего корпуса: 8.6 мин.


### 7.2. Расчет эмбеддингов

Измеренная скорость позволяет закодировать весь локальный корпус, а не только ранее найденных кандидатов. Это дает смысловому поиску возможность находить объявления, пропущенные TF-IDF.


#### Объявления и запросы

Кодируем документы и оба набора запросов. Для запросов используем префикс `query: `. Векторы имеют размерность 384 и хранятся в `float16`. Кодирование корпуса заняло **523 секунды**, массив объявлений - около **378 МБ**.


In [26]:
started = perf_counter()

embedding_dimension = embedding_model.config.hidden_size

document_embeddings = np.empty(
    (len(embedding_corpus), embedding_dimension),
    dtype=np.float16,
)

embedding_text_columns = [
    "item_title_raw",
    "item_infm_params_text",
    "item_description_raw",
]

for start in tqdm(
    range(0, len(embedding_corpus), EMBEDDING_BATCH_SIZE),
    desc="Объявления",
):
    end = min(start + EMBEDDING_BATCH_SIZE, len(embedding_corpus))

    batch_texts = list(map(
        make_embedding_text,
        embedding_corpus.iloc[start:end][
            embedding_text_columns
        ].itertuples(index=False, name=None),
    ))

    document_embeddings[start:end] = encode_text_batch(batch_texts)

print(f"Объявления готовы за {perf_counter() - started:.1f} сек.")


def encode_query_texts(texts):
    result = np.empty(
        (len(texts), embedding_dimension),
        dtype=np.float16,
    )

    for start in tqdm(
        range(0, len(texts), EMBEDDING_BATCH_SIZE),
        desc="Запросы",
    ):
        end = min(start + EMBEDDING_BATCH_SIZE, len(texts))

        batch = [
            "query: " + normalize_query(text)
            for text in texts[start:end]
        ]

        result[start:end] = encode_text_batch(batch)

    return result


valid_embeddings = encode_query_texts(
    valid_queries["search_query"].tolist()
)

benchmark_embeddings = encode_query_texts(
    queries["search_query"].tolist()
)

assert np.isfinite(document_embeddings).all()
assert np.isfinite(valid_embeddings).all()
assert np.isfinite(benchmark_embeddings).all()

print(f"Размер массива объявлений: {document_embeddings.shape}")
print(f"Память массива: {round(document_embeddings.nbytes / 1024**2, 1)} МБ")

Объявления:   0%|          | 0/4031 [00:00<?, ?it/s]

Объявления готовы за 523.4 сек.


Запросы:   0%|          | 0/16 [00:00<?, ?it/s]

Запросы:   0%|          | 0/20 [00:00<?, ?it/s]

Размер массива объявлений: (515895, 384)
Память массива: 377.9 МБ


### 7.3. Смысловое сходство и география

На GPU считаем скалярные произведения нормированных векторов со всем корпусом. Проверяем бонусы совпадения локации 0, 0.05, 0.10 и 0.20; сохраняем по 200 кандидатов.

**Результат:** лучший самостоятельный Recall@50 - **0.688458** при бонусе **0.10**. Без географии он равен 0.288217. Масштаб этих бонусов отличается от TF-IDF, поскольку используются другие оценки сходства.


In [27]:
document_vectors_gpu = torch.from_numpy(
    document_embeddings
).to(embedding_device)

location_ids_gpu = torch.as_tensor(
    corpus_locations,
    dtype=torch.long,
    device=embedding_device,
)

semantic_geo_bonuses = (0.0, 0.05, 0.10, 0.20)
SEMANTIC_TOP_K = 200
SEARCH_BATCH_SIZE = 32


@torch.inference_mode()
def semantic_search(query_vectors, query_locations, corpus_size):
    top_indices = {
        bonus: np.empty(
            (len(query_vectors), SEMANTIC_TOP_K),
            dtype=np.int64,
        )
        for bonus in semantic_geo_bonuses
    }

    documents = document_vectors_gpu[:corpus_size]
    locations = location_ids_gpu[:corpus_size]

    for start in tqdm(
        range(0, len(query_vectors), SEARCH_BATCH_SIZE),
        desc="Смысловой поиск",
    ):
        end = min(start + SEARCH_BATCH_SIZE, len(query_vectors))

        query_batch = torch.from_numpy(
            query_vectors[start:end]
        ).to(embedding_device)

        query_locations_batch = torch.as_tensor(
            query_locations[start:end],
            dtype=torch.long,
            device=embedding_device,
        )

        similarities = (query_batch @ documents.T).float()

        same_location = (
            query_locations_batch[:, None]
            == locations[None, :]
        )

        for bonus in semantic_geo_bonuses:
            scores = similarities + bonus * same_location

            indices = torch.topk(
                scores,
                k=SEMANTIC_TOP_K,
                dim=1,
                largest=True,
                sorted=True,
            ).indices

            top_indices[bonus][start:end] = indices.cpu().numpy()

    return top_indices


started = perf_counter()

semantic_valid_indices = semantic_search(
    valid_embeddings,
    valid_queries["search_location_id"].to_numpy(),
    corpus_size=len(corpus),
)

semantic_only_rows = []

for bonus, predictions_indices in semantic_valid_indices.items():
    recall_50 = []
    recall_200 = []

    for row_number, query in enumerate(
        valid_queries.itertuples(index=False)
    ):
        targets = valid_targets[query.val_id]
        predicted_ids = corpus_ids[predictions_indices[row_number]]

        recall_50.append(
            len(set(predicted_ids[:50]) & targets) / len(targets)
        )
        recall_200.append(
            len(set(predicted_ids) & targets) / len(targets)
        )

    semantic_only_rows.append({
        "geo_bonus": bonus,
        "recall_at_50": np.mean(recall_50),
        "recall_at_200": np.mean(recall_200),
    })

display(pd.DataFrame(semantic_only_rows))
print(f"Время поиска: {perf_counter() - started:.1f} сек.")

Смысловой поиск:   0%|          | 0/63 [00:00<?, ?it/s]

,geo_bonus,recall_at_50,recall_at_200
0,0.00,0.288217,0.443208
1,0.05,0.676208,0.794958
2,0.10,0.688458,0.788458
3,0.20,0.687458,0.781458


Время поиска: 1.9 сек.


### 7.4. Объединение смыслового и предыдущего поиска

Восстанавливаем полный порядок кандидатов варианта с первым CatBoost и объединяем его со смысловыми кандидатами через RRF. Проверяем, дополняют ли друг друга два подхода.

**Результат:** при весе смыслового поиска **0.60** и географическом бонусе **0.10** Recall@50 растет с **0.657046 до 0.716208**. Улучшились 183 запроса, ухудшился 61.


In [28]:
current_order_by_query = {}

groups = rank_valid["group"]
boundaries = np.flatnonzero(
    np.r_[True, groups[1:] != groups[:-1], True]
)

current_blended_scores = (
    0.25 * lexical_rrf
    + 0.75 * catboost_rrf
)

for start, end in zip(boundaries[:-1], boundaries[1:]):
    query_id = int(groups[start])
    candidate_indices = rank_valid["candidate_index"][start:end]

    order = np.lexsort((
        candidate_indices,
        -current_blended_scores[start:end],
    ))

    current_order_by_query[query_id] = candidate_indices[order]


def fuse_candidate_orders(current_order, semantic_order, semantic_weight):
    indices = np.union1d(current_order, semantic_order)
    scores = np.zeros(len(indices), dtype=np.float64)

    current_positions = np.searchsorted(indices, current_order)
    semantic_positions = np.searchsorted(indices, semantic_order)

    scores[current_positions] += (
        (1 - semantic_weight)
        / (60.0 + np.arange(1, len(current_order) + 1))
    )

    scores[semantic_positions] += (
        semantic_weight
        / (60.0 + np.arange(1, len(semantic_order) + 1))
    )

    # не включаем объявления с нулевым вкладом
    active = np.flatnonzero(scores > 0)

    order = np.lexsort((
        indices[active],
        -scores[active],
    ))

    return indices[active[order[:50]]]


semantic_weights = (0.10, 0.25, 0.40, 0.60)

semantic_fusion_rows = []
semantic_fusion_recalls = {}

# считаем контрольный результат прям из восстановленных списков
current_recalls = {}

for query in valid_queries.itertuples(index=False):
    order = current_order_by_query.get(
        query.val_id, np.empty(0, dtype=np.int64)
    )
    targets = valid_targets[query.val_id]

    current_recalls[query.val_id] = (
        len(set(corpus_ids[order[:50]]) & targets) / len(targets)
    )

current_recall_series = pd.Series(current_recalls)

print(f"Контрольный Recall текущего решения: {round(float(current_recall_series.mean()), 6)}")

for bonus in semantic_geo_bonuses:
    for weight in semantic_weights:
        query_recalls = {}

        for row_number, query in enumerate(
            valid_queries.itertuples(index=False)
        ):
            current_order = current_order_by_query.get(
                query.val_id, np.empty(0, dtype=np.int64)
            )

            semantic_order = semantic_valid_indices[bonus][row_number]

            selected = fuse_candidate_orders(
                current_order,
                semantic_order,
                semantic_weight=weight,
            )

            targets = valid_targets[query.val_id]
            query_recalls[query.val_id] = (
                len(set(corpus_ids[selected]) & targets) / len(targets)
            )

        recall_series = pd.Series(query_recalls)
        delta = recall_series - current_recall_series

        semantic_fusion_recalls[(bonus, weight)] = recall_series

        semantic_fusion_rows.append({
            "semantic_geo_bonus": bonus,
            "semantic_weight": weight,
            "recall_at_50": recall_series.mean(),
            "delta_vs_current": delta.mean(),
            "queries_improved": int((delta > 0).sum()),
            "queries_worsened": int((delta < 0).sum()),
        })

semantic_fusion_comparison = (
    pd.DataFrame(semantic_fusion_rows)
    .sort_values("recall_at_50", ascending=False)
    .reset_index(drop=True)
)

display(semantic_fusion_comparison)

Контрольный Recall текущего решения: 0.657046


,semantic_geo_bonus,semantic_weight,recall_at_50,delta_vs_current,queries_improved,queries_worsened
0,0.10,0.60,0.716208,0.059162,183,61
1,0.05,0.60,0.714396,0.057350,182,64
2,0.20,0.60,0.713708,0.056663,183,66
3,0.05,0.40,0.698646,0.041600,114,26
4,0.10,0.40,0.698396,0.041350,113,26
5,0.20,0.40,0.697396,0.040350,113,28
6,0.05,0.25,0.687479,0.030433,77,13
7,0.10,0.25,0.686479,0.029433,74,12
8,0.20,0.25,0.685479,0.028433,74,14
9,0.10,0.10,0.669979,0.012933,33,5


### 7.5. Проверка большего веса E5

Дополнительно проверяем веса смыслового поиска от 0.60 до 1.00. Увеличение его вклада не улучшает результат. Оставляем вес **0.60**, бонус локации **0.10** и локальный Recall@50 **0.716208**.


In [29]:
semantic_refinement_rows = []

for bonus in (0.05, 0.10, 0.20):
    for weight in (0.60, 0.70, 0.80, 0.90, 1.00):
        query_recalls = {}

        for row_number, query in enumerate(
            valid_queries.itertuples(index=False)
        ):
            current_order = current_order_by_query.get(
                query.val_id,
                np.empty(0, dtype=np.int64),
            )

            semantic_order = semantic_valid_indices[bonus][row_number]

            selected = fuse_candidate_orders(
                current_order,
                semantic_order,
                semantic_weight=weight,
            )

            targets = valid_targets[query.val_id]

            query_recalls[query.val_id] = (
                len(set(corpus_ids[selected]) & targets)
                / len(targets)
            )

        recall_series = pd.Series(query_recalls)
        delta = recall_series - current_recall_series

        semantic_refinement_rows.append({
            "semantic_geo_bonus": bonus,
            "semantic_weight": weight,
            "recall_at_50": recall_series.mean(),
            "delta_vs_current": delta.mean(),
            "queries_improved": int((delta > 0).sum()),
            "queries_worsened": int((delta < 0).sum()),
        })

semantic_refinement = (
    pd.DataFrame(semantic_refinement_rows)
    .sort_values(
        ["recall_at_50", "semantic_weight", "semantic_geo_bonus"],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

display(semantic_refinement)

# в проверку включён предыдущий победитель: bonus=0.10, weight=0.60
best_semantic_bonus = float(
    semantic_refinement.iloc[0]["semantic_geo_bonus"]
)
best_semantic_weight = float(
    semantic_refinement.iloc[0]["semantic_weight"]
)

print(f"Географический бонус: {best_semantic_bonus}")
print(f"Вес смыслового поиска: {best_semantic_weight}")

,semantic_geo_bonus,semantic_weight,recall_at_50,delta_vs_current,queries_improved,queries_worsened
0,0.10,0.6,0.716208,0.059162,183,61
1,0.05,0.6,0.714396,0.057350,182,64
2,0.20,0.6,0.713708,0.056663,183,66
3,0.10,0.7,0.707708,0.050662,190,85
4,0.20,0.7,0.705708,0.048663,190,89
5,0.05,0.7,0.703958,0.046913,189,92
6,0.10,0.8,0.697958,0.040913,190,105
7,0.20,0.8,0.695958,0.038913,190,109
8,0.10,0.9,0.695458,0.038413,195,115
9,0.05,0.8,0.695208,0.038163,192,113


Географический бонус: 0.1
Вес смыслового поиска: 0.6


### 7.6. Четвертая отправка

Выполняем смысловой поиск только по объявлениям бенчмарка и объединяем его с полным порядком текстовых кандидатов после первого CatBoost. Используем те же правила, что на валидации. Проверяем и сохраняем новый answer.csv.


In [31]:
started = perf_counter()

n_benchmark_items = len(items)


# смысловой поиск только по корпусу бенчмарка
semantic_benchmark_indices = semantic_search(
    benchmark_embeddings,
    queries["search_location_id"].to_numpy(),
    corpus_size=n_benchmark_items,
)

# сохраняем полные списки кандидатов в памяти (они пригодятся, если потребуется сравнить другое объединение)
benchmark_current_orders = []
semantic_predictions = []

for row_number, query in tqdm(
    enumerate(queries.itertuples(index=False)),
    total=len(queries),
    desc="Объединение поисков",
):
    char_result = get_sparse_similarity(
        Q_char_benchmark,
        row_number,
        char_benchmark_T,
    )
    word_result = get_sparse_similarity(
        Q_word_benchmark,
        row_number,
        word_benchmark_T,
    )

    candidate_indices, features = make_ranker_candidates(
        char_result,
        word_result,
        query.search_location_id,
        normalize_query(query.search_query),
    )

    if len(candidate_indices) > 0:
        lexical_scores = features[
            :, feature_names.index("text_mix_with_geo")
        ]
        model_scores = ranker_model.predict(features)

        lexical_order = np.lexsort((
            candidate_indices,
            -lexical_scores,
        ))
        model_order = np.lexsort((
            candidate_indices,
            -model_scores,
        ))

        lexical_ranks = np.empty(
            len(candidate_indices), dtype=np.float32
        )
        model_ranks = np.empty(
            len(candidate_indices), dtype=np.float32
        )

        lexical_ranks[lexical_order] = np.arange(
            1, len(candidate_indices) + 1
        )
        model_ranks[model_order] = np.arange(
            1, len(candidate_indices) + 1
        )

        current_scores = (
            0.25 / (60.0 + lexical_ranks)
            + 0.75 / (60.0 + model_ranks)
        )

        current_order = candidate_indices[
            np.lexsort((candidate_indices, -current_scores))
        ]
    else:
        current_order = np.empty(0, dtype=np.int64)

    benchmark_current_orders.append(current_order)

    selected = fuse_candidate_orders(
        current_order,
        semantic_benchmark_indices[
            best_semantic_bonus
        ][row_number],
        semantic_weight=best_semantic_weight,
    )

    semantic_predictions.append(corpus_ids[selected].tolist())

answer = pd.DataFrame({
    "query_id": queries["query_id"].to_numpy(),
    "answer": [" ".join(ids) for ids in semantic_predictions],
})

allowed_ids = set(items["item_id"])

assert list(answer.columns) == ["query_id", "answer"]
assert answer["query_id"].is_unique
assert answer["query_id"].tolist() == queries["query_id"].tolist()

for ids in semantic_predictions:
    assert len(ids) <= 50
    assert len(ids) == len(set(ids))
    assert set(ids).issubset(allowed_ids)
    assert all(re.fullmatch(r"[0-9a-f]{16}", value) for value in ids)

answer_path = OUTPUT_DIR / "answer.csv"
answer.to_csv(answer_path, index=False, encoding="utf-8")

saved_answer = pd.read_csv(
    answer_path,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8",
)
pd.testing.assert_frame_equal(saved_answer, answer)

print(f"Файл проверен: {answer_path}")
print(f"Строк: {len(answer)}")
print(f"Запросов с менее чем 50 кандидатами: {sum(len(ids) < 50 for ids in semantic_predictions)}")
print(f"Настройки смыслового поиска: {best_semantic_bonus}, {best_semantic_weight}")
print(f"Время: {perf_counter() - started:.1f} сек.")

Смысловой поиск:   0%|          | 0/77 [00:00<?, ?it/s]

Объединение поисков:   0%|          | 0/2452 [00:00<?, ?it/s]

Файл проверен: /kaggle/working/answer.csv
Строк: 2452
Запросов с менее чем 50 кандидатами: 0
Настройки смыслового поиска: 0.1, 0.6
Время: 44.2 сек.


**Результат отправки 4:** Recall@50 на Stepik - **0.770747**. Добавление смыслового поиска дало наибольший прирост среди последних экспериментов.


## 8. Эксперимент: CatBoost со смысловыми признаками

В предыдущей модели CatBoost не было признаков E5. Теперь расширяем и набор кандидатов, и признаки, чтобы модель могла совместно учитывать текстовые совпадения, смысл и географию.


### 8.1. Смысловые кандидаты для обучения

Кодируем те же 8 000 обучающих запросов и получаем смысловых кандидатов с географическим бонусом 0.10. Для валидации используем уже рассчитанные результаты.


In [32]:
extended_started = perf_counter()

ranker_query_embeddings = encode_query_texts(
    ranker_queries["search_query"].tolist()
)

# функция считает четыре географических варианта (для нового эксперимента используем уже выбранный бонус 0.10)
ranker_semantic_search = semantic_search(
    ranker_query_embeddings,
    ranker_queries["search_location_id"].to_numpy(),
    corpus_size=len(corpus),
)

ranker_semantic_indices = ranker_semantic_search[0.10]

print(f"Смысловые кандидаты для обучения готовы")

Запросы:   0%|          | 0/63 [00:00<?, ?it/s]

Смысловой поиск:   0%|          | 0/250 [00:00<?, ?it/s]

Смысловые кандидаты для обучения готовы


### 8.2. Расширенные кандидаты и признаки

Объединяем до 300 текстовых кандидатов с 200 смысловыми. К прежним 20 признакам добавляем смысловое сходство, его вариант с географическим бонусом и два флага источника кандидата.

**Результат:** Recall набора на валидации вырос с **0.776104 до 0.851333**. В обучении осталось **6 886 групп**: смысловой поиск нашел позитивы для части ранее пропущенных запросов.


In [33]:
extended_feature_names = feature_names + [
    "semantic_similarity",
    "semantic_similarity_with_geo",
    "in_lexical_pool",
    "in_semantic_pool",
]


def make_extended_candidates(
    char_result,
    word_result,
    location,
    query_text,
    query_embedding,
    semantic_indices,
):
    # cохраняем прежний способ генерации текстовых кандидатов
    lexical_indices, _ = make_ranker_candidates(
        char_result,
        word_result,
        location,
        query_text,
    )

    candidate_indices = np.union1d(
        lexical_indices,
        semantic_indices,
    )

    n_candidates = len(candidate_indices)

    char_raw = np.zeros(n_candidates, dtype=np.float32)
    word_raw = np.zeros(n_candidates, dtype=np.float32)

    # получаем текстовые оценки в том числе для новых кандидатов, найденных нейросетью
    _, candidate_positions, result_positions = np.intersect1d(
        candidate_indices,
        char_result.indices,
        return_indices=True,
    )
    char_raw[candidate_positions] = char_result.data[result_positions]

    _, candidate_positions, result_positions = np.intersect1d(
        candidate_indices,
        word_result.indices,
        return_indices=True,
    )
    word_raw[candidate_positions] = word_result.data[result_positions]

    # нормируем по максимумам полного текстового поиска, а не только выбранных кандидатов
    char_max = (
        float(char_result.data.max())
        if len(char_result.data) else 0.0
    )
    word_max = (
        float(word_result.data.max())
        if len(word_result.data) else 0.0
    )

    char_norm = char_raw / max(char_max, 1e-9)
    word_norm = word_raw / max(word_max, 1e-9)

    same_location = (
        corpus_locations[candidate_indices] == location
    )

    text_mix = 0.35 * char_norm + 0.65 * word_norm
    geo_mix = text_mix + 0.80 * same_location
    geo_mix[text_mix <= 0] = 0

    # считаем сходство для всех кандидатов объединённого набора
    selected_vectors = document_embeddings[
        candidate_indices
    ].astype(np.float32)

    semantic_scores = np.einsum(
        "ij,j->i",
        selected_vectors,
        query_embedding.astype(np.float32),
    )

    features = np.column_stack([
        char_raw,
        word_raw,
        char_norm,
        word_norm,
        text_mix,
        geo_mix,
        same_location,
        ranks_in_pool(char_raw),
        ranks_in_pool(word_raw),
        np.full(n_candidates, len(query_text)),
        np.full(n_candidates, len(query_text.split())),
        item_title_lengths[candidate_indices],
        item_numeric_features[candidate_indices],
        semantic_scores,
        semantic_scores + 0.10 * same_location,
        np.isin(candidate_indices, lexical_indices),
        np.isin(candidate_indices, semantic_indices),
    ]).astype(np.float32)

    assert features.shape[1] == len(extended_feature_names)

    return candidate_indices, features


def build_extended_dataset(
    query_frame,
    targets,
    id_column,
    query_embeddings,
    semantic_predictions,
    training=False,
):
    query_char = vectorizer.transform(query_frame["query_norm"])
    query_word = word_vectorizer.transform(query_frame["query_norm"])

    feature_parts = []
    label_parts = []
    group_parts = []
    candidate_parts = []

    pool_recalls = []
    skipped_groups = 0

    for row_number, query in tqdm(
        enumerate(query_frame.itertuples(index=False)),
        total=len(query_frame),
        desc="Расширенные кандидаты",
    ):
        query_id = getattr(query, id_column)

        char_result = get_sparse_similarity(
            query_char, row_number, X_items_T
        )
        word_result = get_sparse_similarity(
            query_word, row_number, X_word_T
        )

        candidate_indices, features = make_extended_candidates(
            char_result,
            word_result,
            query.search_location_id,
            query.query_norm,
            query_embeddings[row_number],
            semantic_predictions[row_number],
        )

        targets_for_query = targets[query_id]

        labels = np.array([
            item_id in targets_for_query
            for item_id in corpus_ids[candidate_indices]
        ], dtype=np.int8)

        pool_recalls.append(
            float(labels.sum()) / len(targets_for_query)
        )

        if training and (
            labels.sum() == 0
            or labels.sum() == len(labels)
        ):
            skipped_groups += 1
            continue

        feature_parts.append(features)
        label_parts.append(labels)
        group_parts.append(
            np.full(len(labels), query_id, dtype=np.int64)
        )
        candidate_parts.append(candidate_indices)

    if not feature_parts:
        raise ValueError("Не удалось собрать обучающие группы.")

    dataset = {
        "X": np.concatenate(feature_parts),
        "y": np.concatenate(label_parts),
        "group": np.concatenate(group_parts),
        "candidate_index": np.concatenate(candidate_parts),
    }

    print("Пар запрос–кандидат:", len(dataset["y"]))
    print("Групп:", len(np.unique(dataset["group"])))
    print("Пропущено обучающих групп:", skipped_groups)
    print(
        "Recall расширенного набора:",
        round(float(np.mean(pool_recalls)), 6),
    )

    return dataset


print("Обучение:")
extended_train = build_extended_dataset(
    ranker_queries,
    ranker_targets,
    id_column="fit_id",
    query_embeddings=ranker_query_embeddings,
    semantic_predictions=ranker_semantic_indices,
    training=True,
)

print("\nВалидация:")
extended_valid = build_extended_dataset(
    valid_queries,
    valid_targets,
    id_column="val_id",
    query_embeddings=valid_embeddings,
    semantic_predictions=semantic_valid_indices[0.10],
    training=False,
)

print(f"\nВремя подготовки: {perf_counter() - extended_started:.1f} сек.")

Обучение:


Расширенные кандидаты:   0%|          | 0/8000 [00:00<?, ?it/s]

Пар запрос–кандидат: 2756618
Групп: 6886
Пропущено обучающих групп: 1114
Recall расширенного набора: 0.857163

Валидация:


Расширенные кандидаты:   0%|          | 0/2000 [00:00<?, ?it/s]

Пар запрос–кандидат: 794912
Групп: 2000
Пропущено обучающих групп: 0
Recall расширенного набора: 0.851333

Время подготовки: 613.3 сек.


### 8.3. Обучение второй модели

Обучаем отдельный CatBoostRanker с теми же основными параметрами и ранней остановкой по NDCG@50. Проверяем Recall@50 на расширенном наборе.

**Результат:** 591 дерево, Recall@50 - **0.709063**. Самостоятельно новая модель уступает варианту четвертой отправки, поэтому далее проверяем их объединение.


In [34]:
extended_train_pool = Pool(
    data=extended_train["X"],
    label=extended_train["y"],
    group_id=extended_train["group"],
    feature_names=extended_feature_names,
)

extended_valid_pool = Pool(
    data=extended_valid["X"],
    label=extended_valid["y"],
    group_id=extended_valid["group"],
    feature_names=extended_feature_names,
)

extended_model = CatBoostRanker(
    iterations=600,
    depth=6,
    learning_rate=0.05,
    loss_function="YetiRank",
    eval_metric="NDCG:top=50",
    task_type="GPU",
    devices="0",
    random_seed=42,
    verbose=100,
    allow_writing_files=False,
)

started = perf_counter()

extended_model.fit(
    extended_train_pool,
    eval_set=extended_valid_pool,
    early_stopping_rounds=80,
    use_best_model=True,
)

extended_valid_scores = extended_model.predict(
    extended_valid_pool
)

extended_model_recalls = evaluate_candidate_scores(
    extended_valid,
    extended_valid_scores,
    valid_targets,
    method="extended_catboost",
)

print(f"Время обучения и предсказания: {perf_counter() - started:.1f} сек.")
print(f"Recall@50 новой модели: {extended_model_recalls.mean()}")

Default metric period is 5 because PFound, NDCG is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=50;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.4023655	best: 0.4023655 (0)	total: 281ms	remaining: 2m 48s
100:	test: 0.5312212	best: 0.5313397 (97)	total: 13.2s	remaining: 1m 5s
200:	test: 0.5430495	best: 0.5432560 (198)	total: 26.1s	remaining: 51.9s
300:	test: 0.5506981	best: 0.5509413 (297)	total: 39.1s	remaining: 38.9s
400:	test: 0.5536617	best: 0.5538885 (382)	total: 52.2s	remaining: 25.9s
500:	test: 0.5557133	best: 0.5557701 (498)	total: 1m 5s	remaining: 12.9s
599:	test: 0.5568369	best: 0.5569035 (590)	total: 1m 18s	remaining: 0us
bestTest = 0.5569035189
bestIteration = 590
Shrink model to first 591 iterations.
Время обучения и предсказания: 80.4 сек.
Recall@50 новой модели: 0.7090625


### 8.4. Объединение с предыдущим решением

Восстанавливаем оценки четвертого варианта на расширенном наборе кандидатов и смешиваем его порядок с порядком новой модели через RRF.

**Лучший вариант:** вес новой модели **0.25**, Recall@50 - **0.719708** против 0.716208 у контроля. Улучшились 17 запросов, ухудшились 9. Небольшой прирост проверяем отдельной отправкой.


In [35]:
extended_groups = extended_valid["group"]
extended_boundaries = np.flatnonzero(np.r_[True, extended_groups[1:] != extended_groups[:-1], True])

# связываем val_id с порядком строк в semantic_valid_indices
valid_row_by_id = {
    int(query_id): row_number
    for row_number, query_id in enumerate(valid_queries["val_id"])
}

previous_rrf_extended = np.zeros(
    len(extended_groups), dtype=np.float64
)
new_model_rrf_extended = np.zeros(
    len(extended_groups), dtype=np.float64
)

for start, end in zip(
    extended_boundaries[:-1],
    extended_boundaries[1:],
):
    query_id = int(extended_groups[start])
    row_number = valid_row_by_id[query_id]

    candidate_indices = extended_valid[
        "candidate_index"
    ][start:end]

    current_order = current_order_by_query.get(
        query_id, np.empty(0, dtype=np.int64)
    )
    semantic_order = semantic_valid_indices[0.10][row_number]

    previous_scores = np.zeros(
        len(candidate_indices), dtype=np.float64
    )

    # все старые кандидаты должны присутствовать в новом наборе
    assert np.isin(current_order, candidate_indices).all()
    assert np.isin(semantic_order, candidate_indices).all()

    previous_scores[
        np.searchsorted(candidate_indices, current_order)
    ] += 0.40 / (
        60.0 + np.arange(1, len(current_order) + 1)
    )

    previous_scores[
        np.searchsorted(candidate_indices, semantic_order)
    ] += 0.60 / (
        60.0 + np.arange(1, len(semantic_order) + 1)
    )

    previous_order = np.lexsort((
        candidate_indices,
        -previous_scores,
    ))
    new_order = np.lexsort((
        candidate_indices,
        -extended_valid_scores[start:end],
    ))

    previous_ranks = np.empty(end - start, dtype=np.float64)
    new_ranks = np.empty(end - start, dtype=np.float64)

    previous_ranks[previous_order] = np.arange(1, end - start + 1)
    new_ranks[new_order] = np.arange(1, end - start + 1)

    previous_rrf_extended[start:end] = (
        1.0 / (60.0 + previous_ranks)
    )
    new_model_rrf_extended[start:end] = (
        1.0 / (60.0 + new_ranks)
    )


extended_comparison_rows = []
extended_recalls_by_weight = {}

for weight in (0.0, 0.25, 0.50, 0.75, 1.0):
    scores = (
        (1 - weight) * previous_rrf_extended
        + weight * new_model_rrf_extended
    )

    recalls = evaluate_candidate_scores(
        extended_valid,
        scores,
        valid_targets,
        method=f"new_model_weight_{weight}",
    )

    extended_recalls_by_weight[weight] = recalls

    if weight == 0:
        reference_recalls = recalls

    delta = recalls - reference_recalls

    extended_comparison_rows.append({
        "new_model_weight": weight,
        "recall_at_50": recalls.mean(),
        "delta_vs_submission_4": delta.mean(),
        "queries_improved": int((delta > 0).sum()),
        "queries_worsened": int((delta < 0).sum()),
    })

extended_comparison = (
    pd.DataFrame(extended_comparison_rows)
    .sort_values(
        ["recall_at_50", "new_model_weight"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(extended_comparison)

,new_model_weight,recall_at_50,delta_vs_submission_4,queries_improved,queries_worsened
0,0.25,0.719708,0.003500,17,9
1,0.50,0.717396,0.001187,32,29
2,0.00,0.716208,0.000000,0,0
3,0.75,0.716146,-0.000063,45,44
4,1.00,0.709063,-0.007146,55,70


### 8.5. Итоговый файл - пятая отправка

Повторяем выбранную схему для бенчмарка: объединяем предыдущий результат и вторую модель с весами **0.75 и 0.25 по обратным рангам**. Возвращаем 50 идентификаторов.

Перед сохранением проверяем строки запросов, отсутствие повторов и принадлежность объявлений корпусу. После записи читаем CSV обратно и сравниваем с исходной таблицей. Эта ячейка формирует итоговый answer.csv.


In [36]:
final_model_weight = 0.25
extended_predictions = []

started = perf_counter()

assert len(benchmark_current_orders) == len(queries)
assert np.array_equal(
    corpus_ids[:len(items)],
    items["item_id"].to_numpy(),
)

for row_number, query in tqdm(
    enumerate(queries.itertuples(index=False)),
    total=len(queries),
    desc="Предсказания новой смеси",
):
    char_result = get_sparse_similarity(
        Q_char_benchmark,
        row_number,
        char_benchmark_T,
    )
    word_result = get_sparse_similarity(
        Q_word_benchmark,
        row_number,
        word_benchmark_T,
    )

    semantic_order = semantic_benchmark_indices[0.10][row_number]

    candidate_indices, features = make_extended_candidates(
        char_result,
        word_result,
        query.search_location_id,
        normalize_query(query.search_query),
        benchmark_embeddings[row_number],
        semantic_order,
    )

    current_order = benchmark_current_orders[row_number]

    assert np.isin(current_order, candidate_indices).all()
    assert np.isin(semantic_order, candidate_indices).all()

    # восстанавливаем оценки четвёртой отправки на полном объединённом наборе кандидатов
    previous_scores = np.zeros(
        len(candidate_indices), dtype=np.float64
    )

    previous_scores[
        np.searchsorted(candidate_indices, current_order)
    ] += 0.40 / (
        60.0 + np.arange(1, len(current_order) + 1)
    )

    previous_scores[
        np.searchsorted(candidate_indices, semantic_order)
    ] += 0.60 / (
        60.0 + np.arange(1, len(semantic_order) + 1)
    )

    new_scores = extended_model.predict(features)

    previous_order = np.lexsort((
        candidate_indices,
        -previous_scores,
    ))
    new_order = np.lexsort((
        candidate_indices,
        -new_scores,
    ))

    previous_ranks = np.empty(
        len(candidate_indices), dtype=np.float64
    )
    new_ranks = np.empty(
        len(candidate_indices), dtype=np.float64
    )

    previous_ranks[previous_order] = np.arange(
        1, len(candidate_indices) + 1
    )
    new_ranks[new_order] = np.arange(
        1, len(candidate_indices) + 1
    )

    final_scores = (
        (1 - final_model_weight) / (60.0 + previous_ranks)
        + final_model_weight / (60.0 + new_ranks)
    )

    selected = np.lexsort((
        candidate_indices,
        -final_scores,
    ))[:50]

    extended_predictions.append(
        corpus_ids[candidate_indices[selected]].tolist()
    )

answer = pd.DataFrame({
    "query_id": queries["query_id"].to_numpy(),
    "answer": [" ".join(ids) for ids in extended_predictions],
})

allowed_ids = set(items["item_id"])

assert list(answer.columns) == ["query_id", "answer"]
assert len(answer) == len(queries)
assert answer["query_id"].is_unique
assert answer["query_id"].tolist() == queries["query_id"].tolist()
assert answer["query_id"].map(
    lambda value: isinstance(value, str) and len(value) == 16
).all()

for ids in extended_predictions:
    assert len(ids) <= 50
    assert len(ids) == len(set(ids))
    assert set(ids).issubset(allowed_ids)
    assert all(re.fullmatch(r"[0-9a-f]{16}", value) for value in ids)

answer_path = OUTPUT_DIR / "answer.csv"
answer.to_csv(answer_path, index=False, encoding="utf-8")

saved_answer = pd.read_csv(
    answer_path,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8",
)
pd.testing.assert_frame_equal(saved_answer, answer)

print(f"Файл проверен: {answer_path}")
print(f"Строк: {len(answer)}")
print(f"Запросов с менее чем 50 кандидатами: {sum(len(ids) < 50 for ids in extended_predictions)}")
print(f"Время: {perf_counter() - started:.1f} сек.")

Предсказания новой смеси:   0%|          | 0/2452 [00:00<?, ?it/s]

Файл проверен: /kaggle/working/answer.csv
Строк: 2452
Запросов с менее чем 50 кандидатами: 0
Время: 55.9 сек.


## 9. Итоги

**Финальный Recall@50 на Stepik - 0.773831.**

| Отправка | Вариант | Локальный Recall@50 | Recall@50 на Stepik |
| --- | --- | ---: | ---: |
| 1 | Два TF-IDF, равные веса, бонус локации 0.60 | 0.627337 | 0.698514 |
| 2 | Вес словесного TF-IDF 0.65, бонус локации 0.80 | 0.649796 | 0.709335 |
| 3 | Текстовый поиск + первый CatBoost | 0.657046 | 0.713304 |
| 4 | Предыдущий вариант + E5 | 0.716208 | 0.770747 |
| 5 | Предыдущий вариант + CatBoost со смысловыми признаками | **0.719708** | **0.773831** |

Наиболее полезными оказались география и сочетание текстового и смыслового поиска. CatBoost дал меньший дополнительный прирост. Итоговая схема использует три последовательных объединения по обратным рангам: текстовый поиск и первый CatBoost с весами 0.25/0.75; этот результат и E5 с весами 0.40/0.60; затем предыдущий результат и второй CatBoost с весами 0.75/0.25. На каждом этапе ранги вычисляются заново, поэтому эти веса нельзя просто перемножить в одну линейную формулу.
